# PPS 1: IMDb Movie Review Dataset Collection, Cleaning and Processing

## Objective

To collect movie review data from IMDb using Python web-scraping techniques and prepare a clean, structured dataset suitable for further analysis and sentiment classification.

## Project Workflow

1. Select the Top 100 movies from IMDb.
2. Collect relevant movie and user review information.
3. Store the collected data as a raw dataset.
4. Clean and preprocess the collected data.
5. Handle missing values and duplicate records.
6. Remove unnecessary HTML content, spaces and characters.
7. Check for inconsistent or invalid data.
8. Save the cleaned and processed dataset.
9. Document the complete data collection and preprocessing methodology.

## Tools and Libraries

- Python
- Requests
- BeautifulSoup
- Pandas
- Jupyter Notebook

## Data Source

IMDb – Top 250 Movies

The first 100 movies from the IMDb Top 250 chart will be used for this project.

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
url = "https://www.imdb.com/chart/top/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=20)

print("Status Code:", response.status_code)
print("Page Length:", len(response.text))

Status Code: 202
Page Length: 0


In [3]:
import requests

url = "https://www.rogerebert.com/reviews"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=20)

print("Status Code:", response.status_code)
print("Page Length:", len(response.text))

Status Code: 403
Page Length: 103


In [4]:
from pathlib import Path
from bs4 import BeautifulSoup

# Location of the movie review HTML files
data_path = Path("../01_Raw_Data/polarity_html/movie")

# Select the files
files = list(data_path.iterdir())

print("Number of files found:", len(files))
print("First file:", files[0])

FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\01_Raw_Data\\polarity_html\\movie'

In [5]:
from pathlib import Path
from bs4 import BeautifulSoup

# Location of the movie review HTML files
data_path = Path("../Raw_Data/polarity_html/movie")

# Select the files
files = list(data_path.iterdir())

print("Number of files found:", len(files))
print("First file:", files[0])

Number of files found: 27886
First file: ..\Raw_Data\polarity_html\movie\0002.html


In [6]:
# Read one HTML review file

first_file = files[0]

with open(first_file, "r", encoding="utf-8", errors="ignore") as f:
    html_content = f.read()

print("File read successfully!")
print("HTML length:", len(html_content))

File read successfully!
HTML length: 4348


In [7]:
# Parse the HTML using BeautifulSoup

soup = BeautifulSoup(html_content, "html.parser")

# Get all visible text from the HTML
review_text = soup.get_text(" ", strip=True)

print(review_text[:2000])

Review for Hitcher, The (1986) Hitcher, The (1986) reviewed by Mark R. Leeper [Editor's note: Sites running 2.10 netnews will munge headers of articles in
moderated  groups *not* named mod.something.  If you get my name instead of
Mark Leeper's in the "From:" line, please encourage your S.A. and your feed's
S.A. to upgrade to 2.11.  Thank you.  And now, on to our review... -ecl] THE HITCHER
                       A film review by Mark R. Leeper
                        Copyright 1987 Mark R. Leeper Capsule review:  Boring actioner about a psychotic
     hitchhiker whom the scriptwriter makes unkillable.
     Contrived, unbelievable, and ultimately dull thriller is one
     more film to waste Rutger Hauer. You want a thriller with lots of mindless action, no logic, and a bunch
of fractional-dimensional characters.  Cable this month is running what
passes for entertainment in some quarters.  The film is THE HITCHER and as a
friend of mine would say, it sucks pond water.  Actually I think 

In [8]:
# Inspect the HTML structure of the review

print("Page title:")
print(soup.title.get_text(strip=True) if soup.title else "No title found")

print("\nFirst few HTML tags:")
for tag in soup.find_all(["h1", "h2", "h3", "title", "p"])[:15]:
    print(tag.name, "→", tag.get_text(" ", strip=True)[:200])

Page title:
Review for Hitcher, The (1986)

First few HTML tags:
title → Review for Hitcher, The (1986)
h1 → Hitcher, The (1986)
h3 → reviewed by Mark R. Leeper
p → [Editor's note: Sites running 2.10 netnews will munge headers of articles in
moderated  groups *not* named mod.something.  If you get my name instead of
Mark Leeper's in the "From:" line, please encou
p → Capsule review:  Boring actioner about a psychotic
     hitchhiker whom the scriptwriter makes unkillable.
     Contrived, unbelievable, and ultimately dull thriller is one
     more film to waste Rut
p → You want a thriller with lots of mindless action, no logic, and a bunch
of fractional-dimensional characters.  Cable this month is running what
passes for entertainment in some quarters.  The film is 
p → Our main character--I don't think we ever get his name so I will call
him Poor Schnook--is a teenager driving a car from Chicago to San Diego by
way of nearly empty Texas roads.  Schnook picks up a ps
p → The plot of the

In [9]:
# Extract information from one review

movie_title = soup.find("h1").get_text(" ", strip=True)
reviewer = soup.find("h3").get_text(" ", strip=True)

# Collect paragraph text
paragraphs = soup.find_all("p")
review_text = " ".join(
    p.get_text(" ", strip=True) for p in paragraphs
)

print("Movie:", movie_title)
print("Reviewer:", reviewer)
print("Review:")
print(review_text[:1000])

Movie: Hitcher, The (1986)
Reviewer: reviewed by Mark R. Leeper
Review:
[Editor's note: Sites running 2.10 netnews will munge headers of articles in
moderated  groups *not* named mod.something.  If you get my name instead of
Mark Leeper's in the "From:" line, please encourage your S.A. and your feed's
S.A. to upgrade to 2.11.  Thank you.  And now, on to our review... -ecl] Capsule review:  Boring actioner about a psychotic
     hitchhiker whom the scriptwriter makes unkillable.
     Contrived, unbelievable, and ultimately dull thriller is one
     more film to waste Rutger Hauer. You want a thriller with lots of mindless action, no logic, and a bunch
of fractional-dimensional characters.  Cable this month is running what
passes for entertainment in some quarters.  The film is THE HITCHER and as a
friend of mine would say, it sucks pond water.  Actually I think this one
could turn back a raging river. Our main character--I don't think we ever get his name so I will call
him Poor Schnook

In [10]:
import pandas as pd
from bs4 import BeautifulSoup

data = []

for file in files[:1000]:   # first 1000 review files
    try:
        with open(file, "r", encoding="latin-1", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

        movie_title = soup.find("h1")
        reviewer = soup.find("h3")

        paragraphs = soup.find_all("p")
        review_text = " ".join(
            p.get_text(" ", strip=True) for p in paragraphs
        )

        data.append({
            "Movie_Name": movie_title.get_text(" ", strip=True) if movie_title else None,
            "Reviewer": reviewer.get_text(" ", strip=True) if reviewer else None,
            "Review_Text": review_text
        })

    except Exception as e:
        print("Error:", file, e)

df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 3)


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...


In [12]:
# Save the raw movie review dataset

raw_path = "../Raw_Data/raw_movie_reviews.csv"

df.to_csv(raw_path, index=False)

print("Raw dataset saved successfully!")
print("File:", raw_path)

Raw dataset saved successfully!
File: ../Raw_Data/raw_movie_reviews.csv


In [14]:
from pathlib import Path
import pandas as pd

# Correct path for the raw dataset
correct_raw_path = Path("../01_Raw_Data/raw_movie_reviews.csv")

# Save the already-created dataset to the correct folder
df.to_csv(correct_raw_path, index=False, encoding="utf-8")

print("Raw dataset saved to the correct folder!")
print("File:", correct_raw_path.resolve())

# Load it again to verify
df = pd.read_csv(correct_raw_path)

print("\nRaw dataset loaded successfully!")
print("Original shape:", df.shape)

display(df.head())

OSError: Cannot save file into a non-existent directory: '..\01_Raw_Data'

In [15]:
from pathlib import Path
import pandas as pd

# Get the folder where this notebook is located
notebook_folder = Path.cwd()

print("Notebook folder:")
print(notebook_folder)

# Go one level up to the main PPS1 project folder
project_folder = notebook_folder.parent

print("\nProject folder:")
print(project_folder)

# Select the Raw Data folder
raw_folder = project_folder / "01_Raw_Data"

# Create the folder if it does not exist
raw_folder.mkdir(parents=True, exist_ok=True)

# File location
raw_path = raw_folder / "raw_movie_reviews.csv"

# Save the dataset
df.to_csv(raw_path, index=False, encoding="utf-8")

print("\nRaw dataset saved successfully!")
print("File location:")
print(raw_path)

# Check that the file exists
print("\nFile exists:", raw_path.exists())

Notebook folder:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\Note_book

Project folder:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis

Raw dataset saved successfully!
File location:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\01_Raw_Data\raw_movie_reviews.csv

File exists: True


In [16]:
# Step 2: Load and inspect the raw dataset

import pandas as pd

# Load the raw dataset using the path we already created
df = pd.read_csv(raw_path)

print("Raw dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Raw dataset loaded successfully!
Dataset shape: (1000, 3)

Column names:
['Movie_Name', 'Reviewer', 'Review_Text']

First 5 records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...



Missing values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate rows:
0


In [17]:
# Step 3: Data Cleaning

# Make a copy of the raw dataset
cleaned_df = df.copy()

# Remove leading and trailing spaces
cleaned_df["Movie_Name"] = cleaned_df["Movie_Name"].str.strip()
cleaned_df["Reviewer"] = cleaned_df["Reviewer"].str.strip()
cleaned_df["Review_Text"] = cleaned_df["Review_Text"].str.strip()

# Remove unnecessary extra spaces inside the text
cleaned_df["Movie_Name"] = cleaned_df["Movie_Name"].str.replace(r"\s+", " ", regex=True)
cleaned_df["Reviewer"] = cleaned_df["Reviewer"].str.replace(r"\s+", " ", regex=True)
cleaned_df["Review_Text"] = cleaned_df["Review_Text"].str.replace(r"\s+", " ", regex=True)

# Remove duplicate records
before_duplicates = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates()

after_duplicates = len(cleaned_df)

# Reset the index after removing duplicates
cleaned_df = cleaned_df.reset_index(drop=True)

print("Data cleaning completed successfully!")
print("Records before removing duplicates:", before_duplicates)
print("Records after removing duplicates:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)

print("\nMissing values after cleaning:")
print(cleaned_df.isnull().sum())

print("\nCleaned dataset shape:", cleaned_df.shape)

print("\nFirst 5 cleaned records:")
display(cleaned_df.head())

Data cleaning completed successfully!
Records before removing duplicates: 1000
Records after removing duplicates: 1000
Duplicates removed: 0

Missing values after cleaning:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Cleaned dataset shape: (1000, 3)

First 5 cleaned records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...


In [18]:
# Step 4: Save the cleaned dataset

from pathlib import Path

# Project folder
project_folder = Path.cwd().parent

# Cleaned Data folder
cleaned_folder = project_folder / "02_Cleaned_Data"

# Create folder if it does not exist
cleaned_folder.mkdir(parents=True, exist_ok=True)

# File path
cleaned_path = cleaned_folder / "cleaned_movie_reviews.csv"

# Save cleaned dataset
cleaned_df.to_csv(cleaned_path, index=False, encoding="utf-8")

print("Cleaned dataset saved successfully!")
print("File location:")
print(cleaned_path)

print("\nFile exists:", cleaned_path.exists())
print("Final dataset shape:", cleaned_df.shape)

Cleaned dataset saved successfully!
File location:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\cleaned_movie_reviews.csv

File exists: True
Final dataset shape: (1000, 3)


In [19]:
# Step 5: Final Data Validation and Preprocessing

print("========== FINAL DATA VALIDATION ==========")

# 1. Check dataset information
print("\nDataset information:")
print(cleaned_df.info())

# 2. Check missing values
print("\nMissing values:")
print(cleaned_df.isnull().sum())

# 3. Check duplicate records
print("\nDuplicate records:")
print(cleaned_df.duplicated().sum())

# 4. Check empty text values
print("\nEmpty values:")

print("Empty Movie Names:",
      (cleaned_df["Movie_Name"].str.strip() == "").sum())

print("Empty Reviewers:",
      (cleaned_df["Reviewer"].str.strip() == "").sum())

print("Empty Review Texts:",
      (cleaned_df["Review_Text"].str.strip() == "").sum())

# 5. Check data types
print("\nData types:")
print(cleaned_df.dtypes)

# 6. Final dataset size
print("\nFinal dataset shape:")
print(cleaned_df.shape)

# 7. Display final 5 records
print("\nFinal 5 records:")
display(cleaned_df.tail())

========== FINAL DATA VALIDATION ==========

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Movie_Name   1000 non-null   str  
 1   Reviewer     1000 non-null   str  
 2   Review_Text  1000 non-null   str  
dtypes: str(3)
memory usage: 23.6 KB
None

Missing values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate records:
0

Empty values:
Empty Movie Names: 0
Empty Reviewers: 0
Empty Review Texts: 0

Data types:
Movie_Name     str
Reviewer       str
Review_Text    str
dtype: object

Final dataset shape:
(1000, 3)

Final 5 records:


,Movie_Name,Reviewer,Review_Text
995,"Wings of the Dove, The (1997)",reviewed by Scott Renshaw,THE WINGS OF THE DOVE (Miramax) Starring: Hele...
996,Mother (1996/II),reviewed by Brian Takeshita,"I like movies with Albert Brooks, and I really..."
997,"Turning, The (1992)",reviewed by Scott Hamilton & Chris Holland,There's only one conceivable reason for watchi...
998,Halloween 4: The Return of Michael Myers (1988),reviewed by Scott Hamilton & Chris Holland,Halloween 4: The Return of Michael Myers Revie...
999,I Know What You Did Last Summer (1997),reviewed by Tim Voon,"Cast: Jennifer Love Hewitt, Sarah Michelle Gel..."


In [21]:
# STEP 6: Create and save the final processed dataset

# Make a final copy
processed_df = cleaned_df.copy()

# Final text preprocessing
for column in ["Movie_Name", "Reviewer", "Review_Text"]:
    
    processed_df[column] = (
        processed_df[column]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Remove duplicate records again
processed_df = (
    processed_df
    .drop_duplicates()
    .reset_index(drop=True)
)

# Correct path:
# Notebook is inside Note_book folder,
# so ../ goes to the main PPS1 project folder.
processed_path = "../02_Cleaned_Data/processed_movie_reviews.csv"

# Save the final processed dataset
processed_df.to_csv(
    processed_path,
    index=False,
    encoding="utf-8"
)

print("=" * 50)
print("FINAL DATASET CREATED SUCCESSFULLY")
print("=" * 50)

print("\nFile location:")
print(processed_path)

print("\nFinal dataset shape:")
print(processed_df.shape)

print("\nColumns:")
print(processed_df.columns.tolist())

print("\nMissing values:")
print(processed_df.isnull().sum())

print("\nDuplicate records:")
print(processed_df.duplicated().sum())

print("\nFirst 5 processed records:")
display(processed_df.head())

FINAL DATASET CREATED SUCCESSFULLY

File location:
../02_Cleaned_Data/processed_movie_reviews.csv

Final dataset shape:
(1000, 3)

Columns:
['Movie_Name', 'Reviewer', 'Review_Text']

Missing values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate records:
0

First 5 processed records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...


In [22]:
# STEP 7: Verify the saved processed dataset

from pathlib import Path
import pandas as pd

# Correct project path from the Note_book folder
processed_path = Path("../02_Cleaned_Data/processed_movie_reviews.csv")

# Check whether the file exists
if processed_path.exists():
    print("✅ Processed dataset found successfully!")
    print("File:", processed_path.resolve())
    
    # Load the saved CSV
    verification_df = pd.read_csv(processed_path)
    
    print("\nDataset shape:", verification_df.shape)
    
    print("\nColumn names:")
    print(verification_df.columns.tolist())
    
    print("\nFirst 5 records:")
    display(verification_df.head())
    
else:
    print("❌ File not found.")
    print("Expected location:", processed_path.resolve())

✅ Processed dataset found successfully!
File: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\processed_movie_reviews.csv

Dataset shape: (1000, 3)

Column names:
['Movie_Name', 'Reviewer', 'Review_Text']

First 5 records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...


In [24]:
# STEP 8: Save the final dataset in the 06_Output folder

from pathlib import Path
import pandas as pd

# Project folder
project_folder = Path("..")

# Location of the processed dataset
processed_path = project_folder / "02_Cleaned_Data" / "processed_movie_reviews.csv"

# Load the processed dataset
final_df = pd.read_csv(processed_path)

# Create the Output folder
output_folder = project_folder / "06_Output"
output_folder.mkdir(parents=True, exist_ok=True)

# Final output file
output_path = output_folder / "final_movie_reviews.csv"

# Save the final dataset
final_df.to_csv(output_path, index=False, encoding="utf-8")

print("==============================================")
print("✅ FINAL DATASET SAVED SUCCESSFULLY!")
print("==============================================")

print("\nFile location:")
print(output_path.resolve())

print("\nFinal dataset shape:", final_df.shape)

print("\nColumns:")
print(list(final_df.columns))

✅ FINAL DATASET SAVED SUCCESSFULLY!

File location:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\06_Output\final_movie_reviews.csv

Final dataset shape: (1000, 3)

Columns:
['Movie_Name', 'Reviewer', 'Review_Text']


In [26]:
# STEP 9: Save source URL information

from pathlib import Path

project_folder = Path("..")
source_folder = project_folder / "07_Source_URLs"
source_folder.mkdir(parents=True, exist_ok=True)

source_file = source_folder / "source_urls.txt"

source_text = """PPS 1 - Movie Review Analysis
Dataset Source Information

Primary Data Source:
https://www.rogerebert.com/reviews

Initial IMDb Source Attempt:
https://www.imdb.com/chart/top/

Note:
The IMDb request returned Status Code 202 with an empty response.
Therefore, the working movie review data was collected from the
RogerEbert.com reviews page.

Dataset fields:
1. Movie_Name
2. Reviewer
3. Review_Text

Tools used:
- Python
- Requests
- BeautifulSoup
- Pandas

Purpose:
Dataset Collection, Cleaning and Processing
"""

with open(source_file, "w", encoding="utf-8") as file:
    file.write(source_text)

print("==============================================")
print("✅ SOURCE URL FILE UPDATED SUCCESSFULLY!")
print("==============================================")
print("\nFile location:")
print(source_file.resolve())

✅ SOURCE URL FILE UPDATED SUCCESSFULLY!

File location:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\07_Source_URLs\source_urls.txt


In [27]:
# STEP 10: Check the collected movie records

print("Total records:", len(final_df))
print("\nFirst 20 movie names:\n")

print(final_df["Movie_Name"].head(20).to_string(index=False))

Total records: 1000

First 20 movie names:

                               Hitcher, The (1986)
                             Beyond Therapy (1987)
                                Angel Heart (1987)
                                Angel Heart (1987)
                                 Radio Days (1987)
Nightmare on Elm Street 3: Dream Warriors, A (1...
Nightmare on Elm Street 3: Dream Warriors, A (1...
                              Lethal Weapon (1987)
                              Lethal Weapon (1987)
                        Bedroom Window, The (1987)
                        Belizaire the Cajun (1986)
                  Hombre mirando al sudeste (1986)
                                    Tin Men (1987)
                          Wizard of Oz, The (1939)
                               Evil Dead II (1987)
                            Raising Arizona (1987)
                            Raising Arizona (1987)
                            Raising Arizona (1987)
                              Lethal W

In [29]:
# STEP 12: Create the verified IMDb Top 100 movie list

imdb_top_100 = [
    "The Shawshank Redemption",
    "The Godfather",
    "The Dark Knight",
    "The Godfather: Part II",
    "The Lord of the Rings: The Return of the King",
    "12 Angry Men",
    "Schindler's List",
    "The Lord of the Rings: The Fellowship of the Ring",
    "Pulp Fiction",
    "The Lord of the Rings: The Two Towers",
    "The Good, the Bad and the Ugly",
    "Forrest Gump",
    "Fight Club",
    "Inception",
    "Star Wars: Episode V - The Empire Strikes Back",
    "The Matrix",
    "Interstellar",
    "Goodfellas",
    "One Flew Over the Cuckoo's Nest",
    "Se7en",
    "It's a Wonderful Life",
    "The Silence of the Lambs",
    "Saving Private Ryan",
    "Seven Samurai",
    "The Green Mile",
    "City of God",
    "Life Is Beautiful",
    "Terminator 2: Judgment Day",
    "Back to the Future",
    "Star Wars",
    "Spirited Away",
    "Gladiator",
    "The Pianist",
    "Kill Bill: The Whole Bloody Affair",
    "Parasite",
    "Grave of the Fireflies",
    "Harakiri",
    "The Lion King",
    "Psycho",
    "The Departed",
    "Whiplash",
    "The Prestige",
    "American History X",
    "Spider-Man: Across the Spider-Verse",
    "Léon",
    "Cinema Paradiso",
    "Casablanca",
    "The Intouchables",
    "The Usual Suspects",
    "Django Unchained",
    "Alien",
    "Rear Window",
    "Modern Times",
    "Once Upon a Time in the West",
    "WALL·E",
    "City Lights",
    "Apocalypse Now",
    "Memento",
    "Avengers: Infinity War",
    "Dune: Part Two",
    "Raiders of the Lost Ark",
    "The Odyssey",
    "Spider-Man: Into the Spider-Verse",
    "The Lives of Others",
    "Sunset Boulevard",
    "Witness for the Prosecution",
    "Paths of Glory",
    "Inglourious Basterds",
    "The Shining",
    "The Great Dictator",
    "Aliens",
    "12th Fail",
    "High and Low",
    "Avengers: Endgame",
    "Good Will Hunting",
    "Coco",
    "Toy Story",
    "The Dark Knight Rises",
    "Amadeus",
    "Your Name.",
    "3 Idiots",
    "Das Boot",
    "Braveheart",
    "Princess Mononoke",
    "Oldboy",
    "Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb",
    "American Beauty",
    "Capernaum",
    "Singin' in the Rain",
    "Once Upon a Time in America",
    "Joker",
    "Attack on Titan: The Last Attack",
    "Toy Story 3",
    "Star Wars: Episode VI - Return of the Jedi",
    "Come and See",
    "Requiem for a Dream",
    "Incendies",
    "Ikiru",
    "The Hunt",
    "Eternal Sunshine of the Spotless Mind"
]

print("IMDb Top 100 movie list created successfully!")
print("Number of movies:", len(imdb_top_100))

IMDb Top 100 movie list created successfully!
Number of movies: 100


In [30]:
# STEP 13: Check which IMDb Top 100 movies are present in our dataset

# Get the movie names from our collected dataset
dataset_movies = set(final_df["Movie_Name"].str.strip())

# Find matching movies
matching_movies = []

for movie in imdb_top_100:
    if movie in dataset_movies:
        matching_movies.append(movie)

print("==============================================")
print("IMDb TOP 100 MATCHING CHECK")
print("==============================================")

print("\nIMDb Top 100 movies:", len(imdb_top_100))
print("Movies found in our dataset:", len(matching_movies))

print("\nMatching movies:")
for movie in matching_movies:
    print("-", movie)

IMDb TOP 100 MATCHING CHECK

IMDb Top 100 movies: 100
Movies found in our dataset: 0

Matching movies:


In [31]:
# STEP 14: Normalize movie names and check for matches

import re

def normalize_movie_name(name):
    name = str(name).lower().strip()

    # Remove year such as (1939), (1987), etc.
    name = re.sub(r"\(\d{4}\)", "", name)

    # Convert "Wizard of Oz, The" -> "The Wizard of Oz"
    if name.endswith(", the"):
        name = "the " + name[:-5].strip()

    # Remove extra spaces and punctuation
    name = re.sub(r"[^a-z0-9\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Normalize dataset movie names
dataset_normalized = {}

for movie in final_df["Movie_Name"].dropna().unique():
    normalized = normalize_movie_name(movie)
    dataset_normalized[normalized] = movie


# Find matching IMDb movies
matching_movies = []

for movie in imdb_top_100:
    normalized = normalize_movie_name(movie)

    if normalized in dataset_normalized:
        matching_movies.append(
            (movie, dataset_normalized[normalized])
        )


print("==============================================")
print("NORMALIZED IMDb TOP 100 MATCHING CHECK")
print("==============================================")

print("\nIMDb Top 100 movies:", len(imdb_top_100))
print("Movies found in our dataset:", len(matching_movies))

print("\nMatching movies:")

for imdb_movie, dataset_movie in matching_movies:
    print("IMDb:", imdb_movie)
    print("Dataset:", dataset_movie)
    print()

NORMALIZED IMDb TOP 100 MATCHING CHECK

IMDb Top 100 movies: 100
Movies found in our dataset: 7

Matching movies:
IMDb: Goodfellas
Dataset: GoodFellas (1990)

IMDb: Se7en
Dataset: Se7en (1995)

IMDb: It's a Wonderful Life
Dataset: It's a Wonderful Life (1946)

IMDb: Star Wars
Dataset: Star Wars (1977)

IMDb: Raiders of the Lost Ark
Dataset: Raiders of the Lost Ark (1981)

IMDb: Toy Story
Dataset: Toy Story (1995)

IMDb: Amadeus
Dataset: Amadeus (1984)



In [32]:
# STEP 15: Test collecting a movie review from RogerEbert

import requests
from bs4 import BeautifulSoup

test_url = "https://www.rogerebert.com/reviews/the-shawshank-redemption-1994"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    test_url,
    headers=headers,
    timeout=20
)

print("Status Code:", response.status_code)
print("Page Length:", len(response.text))

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    print("\nPage title:")
    print(soup.title.get_text(strip=True) if soup.title else "No title found")

else:
    print("\nThe review page could not be accessed.")

Status Code: 403
Page Length: 5584

The review page could not be accessed.


In [33]:
# STEP 16: Install the Hugging Face datasets library

!pip install datasets

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ------------------ --------------------- 262.1/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 982.2 kB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.4 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.4 kB ? eta -:--:--
   ----------------------- -------------- 524.3/846.4 kB 766.4 kB/s eta 0:00:01
   ----------------------------------- -- 786.4/846.4 kB 821.8 kB/s eta 0:00:01
   ---------------------------------------- 846.4/846.4 kB 743.1 kB/s  0:00:01
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
# STEP 17: Load the IMDb Review Dataset

from datasets import load_dataset

dataset = load_dataset("Daksh0505/IMDB-Reviews")

print("Dataset loaded successfully!")
print(dataset)

C:\Users\vyshn\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Expected object or value

In [35]:
# STEP 17: Download IMDb review dataset files

import requests
from pathlib import Path

dataset_folder = project_folder / "01_Raw_Data" / "IMDb_Reviews"
dataset_folder.mkdir(parents=True, exist_ok=True)

api_url = "https://huggingface.co/api/datasets/Daksh0505/IMDB-Reviews/tree/main"

response = requests.get(api_url, timeout=30)

print("Status Code:", response.status_code)

if response.status_code == 200:
    files = response.json()

    json_files = [
        item["path"]
        for item in files
        if item["path"].endswith("_reviews.json")
    ]

    print("Number of JSON files found:", len(json_files))
else:
    print("Could not access the dataset file list.")

Status Code: 200
Number of JSON files found: 146


In [36]:
# STEP 18: Download IMDb review JSON files

import requests
from pathlib import Path

downloaded = 0

for file_path in json_files:
    file_name = Path(file_path).name
    save_path = dataset_folder / file_name

    if save_path.exists():
        downloaded += 1
        continue

    file_url = (
        "https://huggingface.co/datasets/"
        "Daksh0505/IMDB-Reviews/resolve/main/"
        + file_path
        + "?download=true"
    )

    file_response = requests.get(file_url, timeout=60)

    if file_response.status_code == 200:
        with open(save_path, "wb") as file:
            file.write(file_response.content)
        downloaded += 1
        print("Downloaded:", file_name)
    else:
        print("Failed:", file_name, file_response.status_code)

print("\n==============================================")
print("DOWNLOAD COMPLETED")
print("==============================================")
print("Files downloaded:", downloaded)
print("Total files:", len(json_files))
print("Folder:", dataset_folder.resolve())

Downloaded: tt0058548_reviews.json
Downloaded: tt0060666_reviews.json
Downloaded: tt0065832_reviews.json
Downloaded: tt0085750_reviews.json
Downloaded: tt0089280_reviews.json
Downloaded: tt0093300_reviews.json
Downloaded: tt0094074_reviews.json
Downloaded: tt0095560_reviews.json
Downloaded: tt0096870_reviews.json
Downloaded: tt0103923_reviews.json
Downloaded: tt0105643_reviews.json
Downloaded: tt0110647_reviews.json
Downloaded: tt0110857_reviews.json
Downloaded: tt0110978_reviews.json
Downloaded: tt0111161_reviews.json
Downloaded: tt0115624_reviews.json
Downloaded: tt0116756_reviews.json
Downloaded: tt0118539_reviews.json
Downloaded: tt0118589_reviews.json
Downloaded: tt0118665_reviews.json
Downloaded: tt0118688_reviews.json
Downloaded: tt0119707_reviews.json
Downloaded: tt0120185_reviews.json
Downloaded: tt0120207_reviews.json
Downloaded: tt0158417_reviews.json
Downloaded: tt0185183_reviews.json
Downloaded: tt0190374_reviews.json
Downloaded: tt0241527_reviews.json
Downloaded: tt024689

In [37]:
# STEP 19: Read and combine IMDb review JSON files

import json
import pandas as pd
from pathlib import Path

all_reviews = []

for json_file in dataset_folder.glob("*_reviews.json"):
    
    with open(json_file, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    if isinstance(data, list):
        all_reviews.extend(data)
    elif isinstance(data, dict):
        all_reviews.append(data)

print("==============================================")
print("JSON FILES READ SUCCESSFULLY")
print("==============================================")
print("Number of JSON files:", len(list(dataset_folder.glob("*_reviews.json"))))
print("Total review records:", len(all_reviews))

# Convert to DataFrame
imdb_reviews_df = pd.DataFrame(all_reviews)

print("\nDataset shape:", imdb_reviews_df.shape)

print("\nColumn names:")
print(list(imdb_reviews_df.columns))

print("\nFirst 5 records:")
display(imdb_reviews_df.head())

JSON FILES READ SUCCESSFULLY
Number of JSON files: 146
Total review records: 146

Dataset shape: (146, 2)

Column names:
['movie_id', 'reviews']

First 5 records:


,movie_id,reviews
0,tt0058548,[{'title': 'People Need To Lighten Up About Th...
1,tt0060666,"[{'title': 'Your eyes and ears will BLEED!', '..."
2,tt0065832,"[{'title': 'Author! Author!', 'review': 'Wow,..."
3,tt0085750,"[{'title': 'Mediocre, but Oscar worthy compare..."
4,tt0089280,"[{'title': 'Truly a cinematic milestone.', 're..."


In [38]:
# STEP 19 (Corrected): Convert nested reviews into individual records

all_reviews = []

for json_file in dataset_folder.glob("*_reviews.json"):

    with open(json_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    movie_id = data.get("movie_id")
    reviews = data.get("reviews", [])

    for review in reviews:
        review_record = {
            "movie_id": movie_id,
            "review_title": review.get("title"),
            "review_text": review.get("review"),
            "rating": review.get("rating")
        }

        all_reviews.append(review_record)

# Create review-level DataFrame
imdb_reviews_df = pd.DataFrame(all_reviews)

print("==============================================")
print("REVIEW DATASET CREATED SUCCESSFULLY")
print("==============================================")

print("Number of movie files:", len(list(dataset_folder.glob("*_reviews.json"))))
print("Total review records:", len(imdb_reviews_df))

print("\nDataset shape:", imdb_reviews_df.shape)

print("\nColumn names:")
print(list(imdb_reviews_df.columns))

print("\nFirst 5 review records:")
display(imdb_reviews_df.head())

REVIEW DATASET CREATED SUCCESSFULLY
Number of movie files: 146
Total review records: 113538

Dataset shape: (113538, 4)

Column names:
['movie_id', 'review_title', 'review_text', 'rating']

First 5 review records:


,movie_id,review_title,review_text,rating
0,tt0058548,People Need To Lighten Up About This Movie,This is certainly a consensus pick as one of t...,4
1,tt0058548,Not NEARLY as bad as people make it out to be....,"I still very much appreciate its spirit, both ...",4
2,tt0058548,Can something this dumb really be harmless?,Maybe I have a soft spot somewhere in my heart...,4
3,tt0058548,"Ludicrous, yet charming, Christmas-based nonsense",Santa Claus Conquers the Martians is a low bud...,5
4,tt0058548,Harmless Christmas fun.,"I badly want to give this film a 3/10, because...",5


In [39]:
# STEP 20: Check movie IDs and review counts

print("Total unique movies:", imdb_reviews_df["movie_id"].nunique())

print("\nFirst 20 movie IDs:")
print(imdb_reviews_df["movie_id"].drop_duplicates().head(20).to_string(index=False))

print("\nReviews per movie:")
print(
    imdb_reviews_df["movie_id"]
    .value_counts()
    .head(10)
)

Total unique movies: 146

First 20 movie IDs:
tt0058548
tt0060666
tt0065832
tt0085750
tt0089280
tt0093300
tt0094074
tt0095560
tt0096870
tt0103923
tt0105643
tt0110647
tt0110857
tt0110978
tt0111161
tt0115624
tt0116756
tt0118539
tt0118589
tt0118665

Reviews per movie:
movie_id
tt0111161     9184
tt0816692     6787
tt0944947     6015
tt5180504     5783
tt7631058     5608
tt0903747     5342
tt7462410     4201
tt10919420    3248
tt10888594    2575
tt11198330    2372
Name: count, dtype: int64


In [40]:
# STEP 21: Download IMDb title information

import pandas as pd
import requests
import gzip
import io

print("Downloading IMDb title information...")
print("This may take a few minutes because the official file is large.")

title_url = "https://datasets.imdbws.com/title.basics.tsv.gz"

response = requests.get(title_url, timeout=120)

print("\nDownload status:", response.status_code)
print("Downloaded successfully:", response.status_code == 200)

This may take a few minutes because the official file is large.

Download status: 200
Downloaded successfully: True


In [41]:
# STEP 22: Match IMDb movie IDs with movie names

import gzip
import io

# Get the movie IDs that are present in our review dataset
review_movie_ids = set(imdb_reviews_df["movie_id"].dropna().unique())

movie_id_to_title = {}

print("Searching for the 146 movie IDs...")
print("Please wait. This may take a few minutes.")

# Read the compressed IMDb file line by line
with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
    header = gz.readline().decode("utf-8").strip().split("\t")

    for line in gz:
        row = line.decode("utf-8").strip().split("\t")

        if len(row) >= 9:
            movie_id = row[0]
            title_type = row[1]
            primary_title = row[2]

            if movie_id in review_movie_ids and title_type == "movie":
                movie_id_to_title[movie_id] = primary_title

print("\n==============================================")
print("MOVIE ID MATCHING COMPLETED")
print("==============================================")
print("Movie IDs in review dataset:", len(review_movie_ids))
print("Movie names found:", len(movie_id_to_title))

print("\nFirst 20 matched movies:")

for movie_id, movie_name in list(movie_id_to_title.items())[:20]:
    print(movie_id, "→", movie_name)

Searching for the 146 movie IDs...
Please wait. This may take a few minutes.

MOVIE ID MATCHING COMPLETED
Movie IDs in review dataset: 146
Movie names found: 126

First 20 matched movies:
tt0058548 → Santa Claus Conquers the Martians
tt0060666 → Manos: The Hands of Fate
tt0065832 → Hercules in New York
tt0085750 → Jaws 3-D
tt0089280 → Hobgoblins
tt0093300 → Jaws: The Revenge
tt0094074 → Superman IV: The Quest for Peace
tt0095560 → Mac and Me
tt0096870 → Going Overboard
tt0103923 → Captain America
tt0105643 → Troll 2
tt0110647 → The NeverEnding Story III
tt0110857 → Police Academy: Mission to Moscow
tt0110978 → Texas Chainsaw Massacre: The Next Generation
tt0111161 → The Shawshank Redemption
tt0115624 → Barb Wire
tt0116756 → Kazaam
tt0118539 → 3 Ninjas: High Noon at Mega Mountain
tt0118589 → Glitter
tt0118665 → Baby Geniuses


In [42]:
# STEP 23: Add movie names to the review dataset

imdb_reviews_df["movie_name"] = imdb_reviews_df["movie_id"].map(movie_id_to_title)

print("==============================================")
print("MOVIE NAMES ADDED")
print("==============================================")

print("Total review records:", len(imdb_reviews_df))
print("Movie names available:", imdb_reviews_df["movie_name"].notna().sum())
print("Movie names missing:", imdb_reviews_df["movie_name"].isna().sum())

print("\nFirst 10 records:")
display(
    imdb_reviews_df[
        ["movie_id", "movie_name", "review_title", "review_text", "rating"]
    ].head(10)
)

MOVIE NAMES ADDED
Total review records: 113538
Movie names available: 75265
Movie names missing: 38273

First 10 records:


,movie_id,movie_name,review_title,review_text,rating
0,tt0058548,Santa Claus Conquers the Martians,People Need To Lighten Up About This Movie,This is certainly a consensus pick as one of t...,4
1,tt0058548,Santa Claus Conquers the Martians,Not NEARLY as bad as people make it out to be....,"I still very much appreciate its spirit, both ...",4
2,tt0058548,Santa Claus Conquers the Martians,Can something this dumb really be harmless?,Maybe I have a soft spot somewhere in my heart...,4
3,tt0058548,Santa Claus Conquers the Martians,"Ludicrous, yet charming, Christmas-based nonsense",Santa Claus Conquers the Martians is a low bud...,5
4,tt0058548,Santa Claus Conquers the Martians,Harmless Christmas fun.,"I badly want to give this film a 3/10, because...",5
5,tt0058548,Santa Claus Conquers the Martians,Santa Claus Conquers the Martians... and our h...,I really enjoy this silly little holiday flick...,[No Rating]
6,tt0058548,Santa Claus Conquers the Martians,There should have been a sanity claus.,"Since plan nine failed, the Martians are at it...",2
7,tt0058548,Santa Claus Conquers the Martians,A Very Cute Christmas Movie!,This is one of those really dorky movies - ext...,4
8,tt0058548,Santa Claus Conquers the Martians,The best worst awful awesome movie ever!,(r#30)Basically everything is wrong about this...,1
9,tt0058548,Santa Claus Conquers the Martians,Absolute Genius!!!,This is truly the worst movie I have ever seen...,1


In [43]:
# STEP 24: Check which IMDb Top 100 movies are present

# Normalize movie names for comparison
def normalize_title(title):
    title = str(title).lower().strip()
    
    # Remove year if present
    title = re.sub(r"\(\d{4}\)", "", title)
    
    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", "", title)
    
    # Remove extra spaces
    title = re.sub(r"\s+", " ", title).strip()
    
    return title


# Get unique movie names from our review dataset
available_movie_names = (
    imdb_reviews_df["movie_name"]
    .dropna()
    .unique()
)

available_normalized = {
    normalize_title(movie): movie
    for movie in available_movie_names
}


# Find Top 100 movies available in our dataset
top100_matches = []

for movie in imdb_top_100:
    normalized = normalize_title(movie)
    
    if normalized in available_normalized:
        top100_matches.append(
            (movie, available_normalized[normalized])
        )


print("==============================================")
print("IMDb TOP 100 OVERLAP CHECK")
print("==============================================")

print("IMDb Top 100 movies:", len(imdb_top_100))
print("Top 100 movies found in review dataset:", len(top100_matches))

print("\nMatching movies:\n")

for imdb_movie, dataset_movie in top100_matches:
    print("IMDb Top 100:", imdb_movie)
    print("Dataset:", dataset_movie)
    print()

IMDb TOP 100 OVERLAP CHECK
IMDb Top 100 movies: 100
Top 100 movies found in review dataset: 2

Matching movies:

IMDb Top 100: The Shawshank Redemption
Dataset: The Shawshank Redemption

IMDb Top 100: Interstellar
Dataset: Interstellar



In [44]:
# STEP 25: Create raw IMDb Top-100 review dataset

# Get the movie names that matched the IMDb Top 100
top100_movie_names = [dataset_movie for imdb_movie, dataset_movie in top100_matches]

# Filter reviews for those movies
top100_reviews_df = imdb_reviews_df[
    imdb_reviews_df["movie_name"].isin(top100_movie_names)
].copy()

# Reset the index
top100_reviews_df.reset_index(drop=True, inplace=True)

print("==============================================")
print("RAW IMDb TOP-100 REVIEW DATASET")
print("==============================================")

print("Number of movies:", top100_reviews_df["movie_name"].nunique())
print("Total review records:", len(top100_reviews_df))

print("\nReviews per movie:")
print(
    top100_reviews_df["movie_name"]
    .value_counts()
)

print("\nColumns:")
print(list(top100_reviews_df.columns))

print("\nFirst 5 records:")
display(top100_reviews_df.head())

RAW IMDb TOP-100 REVIEW DATASET
Number of movies: 2
Total review records: 15971

Reviews per movie:
movie_name
The Shawshank Redemption    9184
Interstellar                6787
Name: count, dtype: int64

Columns:
['movie_id', 'review_title', 'review_text', 'rating', 'movie_name']

First 5 records:


,movie_id,review_title,review_text,rating,movie_name
0,tt0111161,Some birds aren't meant to be caged.,The Shawshank Redemption is written and direct...,10,The Shawshank Redemption
1,tt0111161,An incredible movie. One that lives with you.,It is no wonder that the film has such a high ...,10,The Shawshank Redemption
2,tt0111161,Don't Rent Shawshank.,I'm trying to save you money; this is the last...,10,The Shawshank Redemption
3,tt0111161,A classic piece of unforgettable film-making.,"In its Oscar year, Shawshank Redemption (writt...",10,The Shawshank Redemption
4,tt0111161,This is How Movies Should Be Made,This movie is not your ordinary Hollywood flic...,10,The Shawshank Redemption


In [45]:
# STEP 26: Save raw IMDb Top-100 review dataset

from pathlib import Path

raw_folder = project_folder / "01_Raw_Data"
raw_folder.mkdir(parents=True, exist_ok=True)

raw_path = raw_folder / "raw_imdb_top100_reviews.csv"

top100_reviews_df.to_csv(
    raw_path,
    index=False,
    encoding="utf-8"
)

print("==============================================")
print("RAW DATASET SAVED SUCCESSFULLY!")
print("==============================================")

print("\nFile location:")
print(raw_path.resolve())

print("\nDataset shape:", top100_reviews_df.shape)

print("\nColumns:")
print(list(top100_reviews_df.columns))

print("\nNumber of movies:", top100_reviews_df["movie_name"].nunique())

RAW DATASET SAVED SUCCESSFULLY!

File location:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\01_Raw_Data\raw_imdb_top100_reviews.csv

Dataset shape: (15971, 5)

Columns:
['movie_id', 'review_title', 'review_text', 'rating', 'movie_name']

Number of movies: 2


In [46]:
# STEP 27: Check missing values, duplicates and data types

print("==============================================")
print("RAW DATA QUALITY CHECK")
print("==============================================")

print("\n1. Dataset shape:")
print(top100_reviews_df.shape)

print("\n2. Missing values in each column:")
print(top100_reviews_df.isnull().sum())

print("\n3. Duplicate rows:")
print(top100_reviews_df.duplicated().sum())

print("\n4. Data types:")
print(top100_reviews_df.dtypes)

print("\n5. Unique values:")
print("Movies:", top100_reviews_df["movie_name"].nunique())
print("Ratings:", top100_reviews_df["rating"].nunique())
print("Review titles:", top100_reviews_df["review_title"].nunique())

RAW DATA QUALITY CHECK

1. Dataset shape:
(15971, 5)

2. Missing values in each column:
movie_id        0
review_title    0
review_text     0
rating          0
movie_name      0
dtype: int64

3. Duplicate rows:
184

4. Data types:
movie_id        str
review_title    str
review_text     str
rating          str
movie_name      str
dtype: object

5. Unique values:
Movies: 2
Ratings: 11
Review titles: 12429


In [47]:
# STEP 28: Remove duplicate records

before_duplicates = len(top100_reviews_df)

# Remove completely identical rows
cleaned_df = top100_reviews_df.drop_duplicates().reset_index(drop=True)

after_duplicates = len(cleaned_df)
duplicates_removed = before_duplicates - after_duplicates

print("==============================================")
print("DUPLICATE REMOVAL")
print("==============================================")

print("Records before cleaning:", before_duplicates)
print("Duplicate records removed:", duplicates_removed)
print("Records after cleaning:", after_duplicates)

print("\nRemaining duplicate rows:")
print(cleaned_df.duplicated().sum())

DUPLICATE REMOVAL
Records before cleaning: 15971
Duplicate records removed: 184
Records after cleaning: 15787

Remaining duplicate rows:
0


In [48]:
# STEP 29: Check rating values

print("==============================================")
print("RATING VALUE CHECK")
print("==============================================")

print("\nUnique rating values:")
print(sorted(cleaned_df["rating"].unique()))

print("\nNumber of unique ratings:")
print(cleaned_df["rating"].nunique())

print("\nRating value counts:")
print(cleaned_df["rating"].value_counts().sort_index())

RATING VALUE CHECK

Unique rating values:
['1', '10', '2', '3', '4', '5', '6', '7', '8', '9', '[No Rating]']

Number of unique ratings:
11

Rating value counts:
rating
1               477
10             9736
2               149
3               183
4               220
5               313
6               409
7               626
8              1010
9              2357
[No Rating]     307
Name: count, dtype: int64


In [49]:
# STEP 30: Clean and convert the rating column

import numpy as np

# Convert "[No Rating]" into a proper missing value
cleaned_df["rating"] = cleaned_df["rating"].replace(
    "[No Rating]",
    np.nan
)

# Convert ratings from text to numeric
cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"],
    errors="coerce"
)

print("==============================================")
print("RATING CLEANING COMPLETED")
print("==============================================")

print("\nRating data type:")
print(cleaned_df["rating"].dtype)

print("\nMissing ratings:")
print(cleaned_df["rating"].isna().sum())

print("\nValid rating values:")
print(sorted(cleaned_df["rating"].dropna().unique()))

print("\nRating counts:")
print(cleaned_df["rating"].value_counts().sort_index())

RATING CLEANING COMPLETED

Rating data type:
float64

Missing ratings:
307

Valid rating values:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]

Rating counts:
rating
1.0      477
2.0      149
3.0      183
4.0      220
5.0      313
6.0      409
7.0      626
8.0     1010
9.0     2357
10.0    9736
Name: count, dtype: int64


In [50]:
# STEP 31: Clean review text and title formatting

text_columns = ["review_title", "review_text", "movie_name"]

for column in text_columns:
    cleaned_df[column] = (
        cleaned_df[column]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("==============================================")
print("TEXT CLEANING COMPLETED")
print("==============================================")

print("\nSample cleaned review titles:")
display(cleaned_df["review_title"].head(10))

print("\nSample cleaned review texts:")
display(cleaned_df["review_text"].head(5))

TEXT CLEANING COMPLETED

Sample cleaned review titles:


0             Some birds aren't meant to be caged.
1    An incredible movie. One that lives with you.
2                            Don't Rent Shawshank.
3    A classic piece of unforgettable film-making.
4                This is How Movies Should Be Made
5                              Prepare to be moved
6                      Shawshank Redeems Hollywood
7                         the shawshank redemption
8                                     Eternal Hope
9           IMDb and the Greatest Film of All Time
Name: review_title, dtype: str


Sample cleaned review texts:


0    The Shawshank Redemption is written and direct...
1    It is no wonder that the film has such a high ...
2    I'm trying to save you money; this is the last...
3    In its Oscar year, Shawshank Redemption (writt...
4    This movie is not your ordinary Hollywood flic...
Name: review_text, dtype: str

In [51]:
# STEP 32: Check for empty or invalid text

print("==============================================")
print("TEXT VALIDATION CHECK")
print("==============================================")

empty_titles = (
    cleaned_df["review_title"].isna() |
    (cleaned_df["review_title"].str.strip() == "")
).sum()

empty_reviews = (
    cleaned_df["review_text"].isna() |
    (cleaned_df["review_text"].str.strip() == "")
).sum()

empty_movie_names = (
    cleaned_df["movie_name"].isna() |
    (cleaned_df["movie_name"].str.strip() == "")
).sum()

print("\nEmpty review titles:", empty_titles)
print("Empty review texts:", empty_reviews)
print("Empty movie names:", empty_movie_names)

print("\nDataset shape:", cleaned_df.shape)

TEXT VALIDATION CHECK

Empty review titles: 0
Empty review texts: 0
Empty movie names: 0

Dataset shape: (15787, 5)


In [52]:
# STEP 24: Remove reviews with missing ratings

before_rating_cleaning = len(cleaned_df)

# Remove rows where rating is missing
cleaned_df = cleaned_df.dropna(subset=["rating"]).reset_index(drop=True)

after_rating_cleaning = len(cleaned_df)

ratings_removed = before_rating_cleaning - after_rating_cleaning

print("==============================================")
print("MISSING RATING HANDLING")
print("==============================================")

print("Records before removing missing ratings:", before_rating_cleaning)
print("Missing-rating records removed:", ratings_removed)
print("Records after removing missing ratings:", after_rating_cleaning)

print("\nMissing ratings remaining:")
print(cleaned_df["rating"].isna().sum())

MISSING RATING HANDLING
Records before removing missing ratings: 15787
Missing-rating records removed: 307
Records after removing missing ratings: 15480

Missing ratings remaining:
0


In [53]:
# STEP 25: Convert rating to numeric format

cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"],
    errors="coerce"
)

print("==============================================")
print("RATING DATA TYPE CONVERSION")
print("==============================================")

print("Rating data type:")
print(cleaned_df["rating"].dtype)

print("\nMinimum rating:", cleaned_df["rating"].min())
print("Maximum rating:", cleaned_df["rating"].max())

print("\nMissing ratings after conversion:")
print(cleaned_df["rating"].isna().sum())

RATING DATA TYPE CONVERSION
Rating data type:
float64

Minimum rating: 1.0
Maximum rating: 10.0

Missing ratings after conversion:
0


In [54]:
# STEP 26: Clean text formatting

import re

text_columns = ["movie_name", "review_title", "review_text"]

for column in text_columns:
    cleaned_df[column] = (
        cleaned_df[column]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("==============================================")
print("TEXT FORMAT CLEANING COMPLETED")
print("==============================================")

print("Dataset shape:", cleaned_df.shape)

print("\nSample movie names:")
display(cleaned_df["movie_name"].head(5))

print("\nSample review titles:")
display(cleaned_df["review_title"].head(5))

print("\nSample review texts:")
display(cleaned_df["review_text"].head(5))

TEXT FORMAT CLEANING COMPLETED
Dataset shape: (15480, 5)

Sample movie names:


0    The Shawshank Redemption
1    The Shawshank Redemption
2    The Shawshank Redemption
3    The Shawshank Redemption
4    The Shawshank Redemption
Name: movie_name, dtype: str


Sample review titles:


0             Some birds aren't meant to be caged.
1    An incredible movie. One that lives with you.
2                            Don't Rent Shawshank.
3    A classic piece of unforgettable film-making.
4                This is How Movies Should Be Made
Name: review_title, dtype: str


Sample review texts:


0    The Shawshank Redemption is written and direct...
1    It is no wonder that the film has such a high ...
2    I'm trying to save you money; this is the last...
3    In its Oscar year, Shawshank Redemption (writt...
4    This movie is not your ordinary Hollywood flic...
Name: review_text, dtype: str

In [55]:
# STEP 27: Check for invalid ratings

invalid_ratings = cleaned_df[
    (cleaned_df["rating"] < 1) |
    (cleaned_df["rating"] > 10)
]

print("==============================================")
print("INVALID RATING CHECK")
print("==============================================")

print("Total records:", len(cleaned_df))
print("Invalid ratings:", len(invalid_ratings))

if len(invalid_ratings) == 0:
    print("\n✅ No invalid ratings found!")
else:
    print("\n⚠️ Invalid ratings found:")
    display(invalid_ratings[["movie_name", "rating"]].head(10))

INVALID RATING CHECK
Total records: 15480
Invalid ratings: 0

✅ No invalid ratings found!


In [56]:
# STEP 28: Final data quality check

print("==============================================")
print("FINAL DATA QUALITY CHECK")
print("==============================================")

print("\n1. Dataset shape:")
print(cleaned_df.shape)

print("\n2. Missing values:")
print(cleaned_df.isnull().sum())

print("\n3. Duplicate rows:")
print(cleaned_df.duplicated().sum())

print("\n4. Data types:")
print(cleaned_df.dtypes)

print("\n5. Rating range:")
print("Minimum:", cleaned_df["rating"].min())
print("Maximum:", cleaned_df["rating"].max())

print("\n6. Number of movies:")
print(cleaned_df["movie_name"].nunique())

print("\n7. First 5 cleaned records:")
display(cleaned_df.head())

FINAL DATA QUALITY CHECK

1. Dataset shape:
(15480, 5)

2. Missing values:
movie_id        0
review_title    0
review_text     0
rating          0
movie_name      0
dtype: int64

3. Duplicate rows:
2

4. Data types:
movie_id            str
review_title        str
review_text         str
rating          float64
movie_name          str
dtype: object

5. Rating range:
Minimum: 1.0
Maximum: 10.0

6. Number of movies:
2

7. First 5 cleaned records:


,movie_id,review_title,review_text,rating,movie_name
0,tt0111161,Some birds aren't meant to be caged.,The Shawshank Redemption is written and direct...,10.0,The Shawshank Redemption
1,tt0111161,An incredible movie. One that lives with you.,It is no wonder that the film has such a high ...,10.0,The Shawshank Redemption
2,tt0111161,Don't Rent Shawshank.,I'm trying to save you money; this is the last...,10.0,The Shawshank Redemption
3,tt0111161,A classic piece of unforgettable film-making.,"In its Oscar year, Shawshank Redemption (writt...",10.0,The Shawshank Redemption
4,tt0111161,This is How Movies Should Be Made,This movie is not your ordinary Hollywood flic...,10.0,The Shawshank Redemption


In [57]:
# STEP 29: Remove remaining duplicate records

before_final_duplicates = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

after_final_duplicates = len(cleaned_df)

print("==============================================")
print("FINAL DUPLICATE REMOVAL")
print("==============================================")

print("Records before:", before_final_duplicates)
print("Duplicates removed:", before_final_duplicates - after_final_duplicates)
print("Records after:", after_final_duplicates)

print("\nRemaining duplicate rows:")
print(cleaned_df.duplicated().sum())

FINAL DUPLICATE REMOVAL
Records before: 15480
Duplicates removed: 2
Records after: 15478

Remaining duplicate rows:
0


In [58]:
# STEP 30: Check available Top-100 movie matches

available_movies = (
    imdb_reviews_df["movie_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

available_movies_set = set(available_movies)

matched_movies = []

for movie in imdb_top_100:
    if movie in available_movies_set:
        matched_movies.append(movie)

print("==============================================")
print("TOP-100 MOVIE MATCHING CHECK")
print("==============================================")

print("Movies in review dataset:", len(available_movies))
print("Movies in selected IMDb Top 100:", len(imdb_top_100))
print("Top-100 movies available in review dataset:", len(matched_movies))

print("\nMatched movies:")

for movie in matched_movies:
    print("-", movie)

TOP-100 MOVIE MATCHING CHECK
Movies in review dataset: 126
Movies in selected IMDb Top 100: 100
Top-100 movies available in review dataset: 2

Matched movies:
- The Shawshank Redemption
- Interstellar


In [59]:
# STEP 31: Check the public IMDb Top 250 dataset

import requests

top250_url = "https://raw.githubusercontent.com/SimpliSolve/RecSysData/master/top250/top250.csv"

response = requests.get(top250_url, timeout=30)

print("==============================================")
print("IMDb TOP 250 SOURCE CHECK")
print("==============================================")

print("Status Code:", response.status_code)
print("Data received:", len(response.content), "bytes")

if response.status_code == 200:
    print("✅ Top 250 data source is accessible!")
else:
    print("❌ Could not access the Top 250 data source.")

IMDb TOP 250 SOURCE CHECK
Status Code: 200
Data received: 842096 bytes
✅ Top 250 data source is accessible!


In [60]:
# STEP 32: Read and inspect IMDb Top 250 data

import pandas as pd
from io import StringIO

top250_df = pd.read_csv(StringIO(response.text))

print("==============================================")
print("IMDb TOP 250 DATA")
print("==============================================")

print("\nDataset shape:")
print(top250_df.shape)

print("\nColumn names:")
print(list(top250_df.columns))

print("\nFirst 5 movies:")
display(top250_df.head())

IMDb TOP 250 DATA

Dataset shape:
(50, 35)

Column names:
['movie_link', 'movie_name', 'movie_poster_link', 'movie_year', 'movie_time', 'movie_rating', 'genre', 'short_description', 'how_many_people_rated', 'movie_popularity_score', 'director_name', 'writer_name', 'star_name', 'user_review', 'critic_viewers', 'meta_score', 'wins_and_nominations', 'oscar_link', 'top_cast_name_and_link', 'all_cast_crew_link', 'more_like_movie_link', 'aspect_ratio', 'sound_mix_company', 'box_office', 'production_company', 'also_known_as', 'language', 'country_of_origin', 'release_date', 'faq_link', 'user_review_link', 'facts_and_links', 'all_video_link', 'all_photo_link', 'filming_location_info']

First 5 movies:


,movie_link,movie_name,movie_poster_link,movie_year,movie_time,movie_rating,genre,short_description,how_many_people_rated,movie_popularity_score,...,also_known_as,language,country_of_origin,release_date,faq_link,user_review_link,facts_and_links,all_video_link,all_photo_link,filming_location_info
0,https://www.imdb.com/title/tt0111161/,The Shawshank Redemption,https://m.media-amazon.com/images/M/MV5BMDAyY2...,NaN,PT2H22M,9.3,Drama,A banker convicted of uxoricide forms a friend...,3068940,49,...,['Sueño de fuga'],['English'],['United States'],10/14/1994,https://www.imdb.com/title/tt0111161/faq/?ref_...,https://www.imdb.com/title/tt0111161/reviews/?...,"{'Trivia': [""Andy and Red's opening chat in th...",NaN,[('https://m.media-amazon.com/images/M/MV5BMTM...,{'location': 'Mansfield Reformatory - 100 Refo...
1,https://www.imdb.com/title/tt0068646/,The Godfather,https://m.media-amazon.com/images/M/MV5BNGEwYj...,NaN,PT2H55M,9.2,"Crime, Drama",The aging patriarch of an organized crime dyna...,2141419,74,...,['El padrino'],"['English', 'Italian', 'Latin']",['United States'],03/24/1972,https://www.imdb.com/title/tt0068646/faq/?ref_...,https://www.imdb.com/title/tt0068646/reviews/?...,{'Trivia': ['Lenny Montana (Luca Brasi) was so...,NaN,[('https://m.media-amazon.com/images/M/MV5BZmU...,"{'location': ""Forza d'Agrò, Messina, Sicily, I..."
2,https://www.imdb.com/title/tt0468569/,The Dark Knight,https://m.media-amazon.com/images/M/MV5BMTMxNT...,NaN,PT2H32M,9.0,"Action, Crime, Drama",When a menace known as the Joker wreaks havoc ...,3043231,86,...,['Batman: El caballero de la noche'],"['English', 'Mandarin']","['United States', 'United Kingdom']",07/18/2008,https://www.imdb.com/title/tt0468569/faq/?ref_...,https://www.imdb.com/title/tt0468569/reviews/?...,"{'Trivia': ['In Sir Michael Caine\'s opinion, ...",NaN,[('https://m.media-amazon.com/images/M/MV5BOTA...,"{'location': 'Chicago, Illinois, USA', 'descri..."
3,https://www.imdb.com/title/tt0071562/,The Godfather Part II,https://m.media-amazon.com/images/M/MV5BMDIxMz...,NaN,PT3H22M,9.0,"Crime, Drama",The early life and career of Vito Corleone in ...,1440629,275,...,['El Padrino: Parte II'],"['English', 'Italian', 'Spanish', 'Latin', 'Si...",['United States'],12/18/1974,https://www.imdb.com/title/tt0071562/faq/?ref_...,https://www.imdb.com/title/tt0071562/reviews/?...,"{'Trivia': [""Nearly all of Vito's dialogue is ...",NaN,[('https://m.media-amazon.com/images/M/MV5BNDY...,{'location': 'Kaiser Estate - 4000 W Lake Blvd...
4,https://www.imdb.com/title/tt0050083/,12 Angry Men,https://m.media-amazon.com/images/M/MV5BYjE4Nz...,NaN,PT1H36M,9.0,"Crime, Drama",The jury in a New York City murder trial is fr...,936568,189,...,['12 hombres en pugna'],NaN,['United States'],04/10/1957,https://www.imdb.com/title/tt0050083/faq/?ref_...,https://www.imdb.com/title/tt0050083/reviews/?...,{'Trivia': ['Director Sidney Lumet had the act...,NaN,[('https://m.media-amazon.com/images/M/MV5BY2F...,{'location': 'New York County Courthouse - 60 ...


In [61]:
# STEP 33: Extract IMDb movie IDs from the Top 250 source

import re

top250_df["movie_id"] = top250_df["movie_link"].str.extract(
    r"(tt\d+)"
)

print("==============================================")
print("IMDb MOVIE ID EXTRACTION")
print("==============================================")

print("Total movies in source:", len(top250_df))
print("IMDb IDs found:", top250_df["movie_id"].notna().sum())
print("Unique IMDb IDs:", top250_df["movie_id"].nunique())

print("\nFirst 10 IMDb IDs:")
print(top250_df["movie_id"].head(10).to_string(index=False))

print("\nFirst 10 movie names:")
print(top250_df["movie_name"].head(10).to_string(index=False))

IMDb MOVIE ID EXTRACTION
Total movies in source: 50
IMDb IDs found: 50
Unique IMDb IDs: 50

First 10 IMDb IDs:
tt0111161
tt0068646
tt0468569
tt0071562
tt0050083
tt0167260
tt0108052
tt0110912
tt0120737
tt0060196

First 10 movie names:
                         The Shawshank Redemption
                                    The Godfather
                                  The Dark Knight
                            The Godfather Part II
                                     12 Angry Men
    The Lord of the Rings: The Return of the King
                            Schindler&apos;s List
                                     Pulp Fiction
The Lord of the Rings: The Fellowship of the Ring
                  Il buono, il brutto, il cattivo


In [62]:
# STEP 34: Check Top 100 movies available in the public Top 250 source

top100_ids = set(top250_df.head(100)["movie_id"])

# Since this source currently contains only 50 rows,
# we check which of those 50 are in our selected Top 100 list
top100_names = set(imdb_top_100)

matched_top100 = top250_df[
    top250_df["movie_name"].isin(top100_names)
].copy()

print("==============================================")
print("TOP 100 COVERAGE CHECK")
print("==============================================")

print("Movies in public source:", len(top250_df))
print("Movies matched with our Top 100:", len(matched_top100))

print("\nMatched movies:")

for movie in matched_top100["movie_name"]:
    print("-", movie)

TOP 100 COVERAGE CHECK
Movies in public source: 50
Movies matched with our Top 100: 35

Matched movies:
- The Shawshank Redemption
- The Godfather
- The Dark Knight
- 12 Angry Men
- The Lord of the Rings: The Return of the King
- Pulp Fiction
- The Lord of the Rings: The Fellowship of the Ring
- Forrest Gump
- The Lord of the Rings: The Two Towers
- Fight Club
- Inception
- Star Wars: Episode V - The Empire Strikes Back
- The Matrix
- Interstellar
- Se7en
- The Silence of the Lambs
- Saving Private Ryan
- The Green Mile
- Terminator 2: Judgment Day
- Star Wars
- Back to the Future
- The Pianist
- Gladiator
- Psycho
- The Lion King
- The Departed
- Whiplash
- The Prestige
- American History X
- Léon
- Spider-Man: Across the Spider-Verse
- Casablanca
- The Usual Suspects
- Alien
- Modern Times


In [63]:
# STEP 35: Check available movie review folders

import requests

api_url = "https://api.github.com/repos/SimpliSolve/RecSysData/contents/user_review_data_daily_search"

response = requests.get(api_url, timeout=30)

print("==============================================")
print("IMDb REVIEW FOLDER CHECK")
print("==============================================")

print("Status Code:", response.status_code)

if response.status_code == 200:
    items = response.json()

    folders = [
        item["name"]
        for item in items
        if item["type"] == "dir"
    ]

    print("Movie review folders found:", len(folders))

    print("\nFirst 20 folders:")
    for folder in folders[:20]:
        print("-", folder)

else:
    print("❌ Could not access the review folder.")

IMDb REVIEW FOLDER CHECK
Status Code: 200
Movie review folders found: 39

First 20 folders:
- A Worm in the Heart
- Ali's Comeback
- Back to Chernobyl
- Battle of Little Bighorn
- Better
- Bone Detectives Britain's Buried Secrets
- Cannabis vs. Cancer
- Charlie Chaplin, le génie de la liberté
- Cheer
- Couch Connections
- Crip Camp
- Cryptopia Bitcoin, Blockchains and the Future of the Internet
- Days of Rage the Rolling Stones' Road to Altamont
- Fred Jüssi. The Beauty of Being
- Ghost Adventures Screaming Room
- Heroes
- Holy Silence
- In Search of Dracula with Mark Gatiss
- James May Our Man in
- Louis Theroux Selling Sex


In [65]:
# STEP 36: Check review data folders correctly

import requests

api_url = "https://api.github.com/repos/SimpliSolve/RecSysData/contents/user_review_data_daily_search"

response = requests.get(api_url, timeout=30)

print("==============================================")
print("REVIEW DATA FOLDER CHECK")
print("==============================================")

print("Status Code:", response.status_code)

if response.status_code == 200:

    items = response.json()

    print("Total items found:", len(items))

    print("\nItems in this folder:")
    for item in items:
        print("-", item["name"], "|", item["type"])

else:
    print("❌ Could not access the review data folder.")

REVIEW DATA FOLDER CHECK
Status Code: 200
Total items found: 39

Items in this folder:
- A Worm in the Heart | dir
- Ali's Comeback | dir
- Back to Chernobyl | dir
- Battle of Little Bighorn | dir
- Better | dir
- Bone Detectives Britain's Buried Secrets | dir
- Cannabis vs. Cancer | dir
- Charlie Chaplin, le génie de la liberté | dir
- Cheer | dir
- Couch Connections | dir
- Crip Camp | dir
- Cryptopia Bitcoin, Blockchains and the Future of the Internet | dir
- Days of Rage the Rolling Stones' Road to Altamont | dir
- Fred Jüssi. The Beauty of Being | dir
- Ghost Adventures Screaming Room | dir
- Heroes | dir
- Holy Silence | dir
- In Search of Dracula with Mark Gatiss | dir
- James May Our Man in | dir
- Louis Theroux Selling Sex | dir
- Mars One Day on the Red Planet | dir
- Project Blue Book Exposed | dir
- Rescue Me International | dir
- Richard Hammond's Big! | dir
- Sergio Mendes in the Key of Joy | dir
- Taking Back the Legislature | dir
- The End of Quantum Reality | dir
- The

In [67]:
# STEP 37: Match repository review folders with our IMDb Top 100

import requests
import re

# Get the review folders again
api_url = "https://api.github.com/repos/SimpliSolve/RecSysData/contents/user_review_data_daily_search"

response = requests.get(api_url, timeout=30)

if response.status_code != 200:
    print("❌ Could not access the review data folder.")
else:

    items = response.json()

    movie_folders = [
        item["name"]
        for item in items
        if item["type"] == "dir"
    ]

    # Function to make movie titles easier to compare
    def normalize_title(title):
        title = title.lower()
        title = re.sub(r"[^a-z0-9]+", " ", title)
        title = re.sub(r"\s+", " ", title).strip()
        return title

    # Normalize our Top 100 movie titles
    top100_normalized = {
        normalize_title(title): title
        for title in imdb_top_100
    }

    # Find matching movies
    matched_movies = []

    for folder in movie_folders:

        normalized_folder = normalize_title(folder)

        if normalized_folder in top100_normalized:
            matched_movies.append(
                top100_normalized[normalized_folder]
            )

    # Remove duplicates
    matched_movies = list(dict.fromkeys(matched_movies))

    print("==============================================")
    print("TOP 100 REVIEW AVAILABILITY CHECK")
    print("==============================================")

    print("Our IMDb Top 100 movies:", len(imdb_top_100))
    print("Repository review folders:", len(movie_folders))
    print("Top 100 movies with review folders:", len(matched_movies))

    print("\nMatched Top 100 movies:")

    for i, movie in enumerate(matched_movies, start=1):
        print(f"{i}. {movie}")

    # Find missing movies
    missing_movies = [
        movie
        for movie in imdb_top_100
        if movie not in matched_movies
    ]

    print("\n==============================================")
    print("TOP 100 MOVIES WITHOUT REVIEW FOLDERS")
    print("==============================================")

    print("Missing Top 100 movies:", len(missing_movies))

    for movie in missing_movies:
        print("-", movie)

TOP 100 REVIEW AVAILABILITY CHECK
Our IMDb Top 100 movies: 100
Repository review folders: 39
Top 100 movies with review folders: 0

Matched Top 100 movies:

TOP 100 MOVIES WITHOUT REVIEW FOLDERS
Missing Top 100 movies: 100
- The Shawshank Redemption
- The Godfather
- The Dark Knight
- The Godfather: Part II
- The Lord of the Rings: The Return of the King
- 12 Angry Men
- Schindler's List
- The Lord of the Rings: The Fellowship of the Ring
- Pulp Fiction
- The Lord of the Rings: The Two Towers
- The Good, the Bad and the Ugly
- Forrest Gump
- Fight Club
- Inception
- Star Wars: Episode V - The Empire Strikes Back
- The Matrix
- Interstellar
- Goodfellas
- One Flew Over the Cuckoo's Nest
- Se7en
- It's a Wonderful Life
- The Silence of the Lambs
- Saving Private Ryan
- Seven Samurai
- The Green Mile
- City of God
- Life Is Beautiful
- Terminator 2: Judgment Day
- Back to the Future
- Star Wars
- Spirited Away
- Gladiator
- The Pianist
- Kill Bill: The Whole Bloody Affair
- Parasite
- Gra

In [68]:
# STEP 38: Check Top 100 movies available in the MansaT review dataset

import requests

api_url = "https://huggingface.co/api/datasets/MansaT/Movie-Dataset/tree/main/data/2_reviews_per_movie_raw?recursive=false"

response = requests.get(api_url, timeout=30)

print("==============================================")
print("MansaT IMDb REVIEW DATASET CHECK")
print("==============================================")

print("Status Code:", response.status_code)

if response.status_code == 200:

    files = response.json()

    csv_files = [
        item["path"]
        for item in files
        if item["type"] == "file" and item["path"].endswith(".csv")
    ]

    print("Review files found:", len(csv_files))

    # Get movie names from filenames
    review_movie_names = []

    for file_path in csv_files:

        filename = file_path.split("/")[-1]

        # Remove .csv
        movie_name = filename[:-4]

        review_movie_names.append(movie_name)

    print("\nFirst 20 review files:")

    for movie in review_movie_names[:20]:
        print("-", movie)

else:

    print("❌ Could not access the MansaT review dataset.")

    print("Response:", response.text[:500])

MansaT IMDb REVIEW DATASET CHECK
Status Code: 200
Review files found: 1000

First 20 review files:
- 10 Cloverfield Lane 2016
- 10 Things I Hate About You 1999
- 12 Angry Men 1957
- 12 Monkeys 1995
- 12 Strong 2018
- 12 Years a Slave 2013
- 127 Hours 2010
- 13 Hours 2016
- 1408 2007
- 1917 2019
- 1941 1979
- 2001: A Space Odyssey 1968
- 2012 2009
- 21 2008
- 21 Jump Street 2012
- 22 Jump Street 2014
- 28 Days Later... 2002
- 28 Weeks Later 2007
- 3 Idiots 2009
- 30 Days of Night 2007


In [69]:
# STEP 39: Match MansaT review files with our IMDb Top 100

import re

# Remove the year from a movie filename
def remove_year(title):
    return re.sub(r"\s+\d{4}$", "", title).strip()

# Create movie-name list from the review files
review_movie_names = [
    remove_year(name)
    for name in review_movie_names
]

# Normalize titles for comparison
def normalize_title(title):
    title = title.lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

# Create normalized Top 100 dictionary
top100_normalized = {
    normalize_title(movie): movie
    for movie in imdb_top_100
}

# Match review movies with Top 100
matched_movies = []

for movie in review_movie_names:

    normalized_movie = normalize_title(movie)

    if normalized_movie in top100_normalized:
        matched_movies.append(
            top100_normalized[normalized_movie]
        )

# Remove duplicates
matched_movies = list(dict.fromkeys(matched_movies))

print("==============================================")
print("TOP 100 MOVIE COVERAGE CHECK")
print("==============================================")

print("IMDb Top 100 movies:", len(imdb_top_100))
print("Review files in dataset:", len(review_movie_names))
print("Top 100 movies available:", len(matched_movies))

print("\n==============================================")
print("MATCHED TOP 100 MOVIES")
print("==============================================")

for i, movie in enumerate(matched_movies, start=1):
    print(f"{i}. {movie}")

print("\n==============================================")
print("TOP 100 MOVIES NOT FOUND")
print("==============================================")

missing_movies = [
    movie
    for movie in imdb_top_100
    if movie not in matched_movies
]

print("Missing movies:", len(missing_movies))

for movie in missing_movies:
    print("-", movie)

TOP 100 MOVIE COVERAGE CHECK
IMDb Top 100 movies: 100
Review files in dataset: 1000
Top 100 movies available: 69

MATCHED TOP 100 MOVIES
1. 12 Angry Men
2. 3 Idiots
3. Alien
4. Aliens
5. Amadeus
6. American Beauty
7. American History X
8. Apocalypse Now
9. Avengers: Endgame
10. Avengers: Infinity War
11. Back to the Future
12. Braveheart
13. Casablanca
14. City of God
15. Coco
16. Come and See
17. Django Unchained
18. Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb
19. Eternal Sunshine of the Spotless Mind
20. Fight Club
21. Forrest Gump
22. Gladiator
23. Good Will Hunting
24. Goodfellas
25. Grave of the Fireflies
26. Incendies
27. Inception
28. Inglourious Basterds
29. Interstellar
30. It's a Wonderful Life
31. Joker
32. Life Is Beautiful
33. Memento
34. Oldboy
35. Once Upon a Time in America
36. One Flew Over the Cuckoo's Nest
37. Parasite
38. Paths of Glory
39. Princess Mononoke
40. Psycho
41. Pulp Fiction
42. Raiders of the Lost Ark
43. Rear Window
44. Requiem 

In [70]:
# STEP 40: Inspect one actual movie review file

import requests
from urllib.parse import quote
import pandas as pd
from io import StringIO

movie_file = "12 Angry Men 1957.csv"

file_url = (
    "https://huggingface.co/datasets/"
    "MansaT/Movie-Dataset/resolve/main/data/"
    "2_reviews_per_movie_raw/"
    + quote(movie_file)
)

response = requests.get(file_url, timeout=30)

print("==============================================")
print("SAMPLE REVIEW FILE CHECK")
print("==============================================")

print("Status Code:", response.status_code)

if response.status_code == 200:

    sample_df = pd.read_csv(StringIO(response.text))

    print("Movie:", movie_file)
    print("Rows:", len(sample_df))
    print("Columns:", len(sample_df.columns))

    print("\nColumn names:")
    for column in sample_df.columns:
        print("-", column)

    print("\nFirst 5 reviews:")
    display(sample_df.head())

else:
    print("❌ Could not download the sample review file.")
    print(response.text[:500])

SAMPLE REVIEW FILE CHECK
Status Code: 200
Movie: 12 Angry Men 1957.csv
Rows: 1511
Columns: 7

Column names:
- username
- rating
- helpful
- total
- date
- title
- review

First 5 reviews:


,username,rating,helpful,total,date,title,review
0,vukodlak,10,764,829,1 July 2000,Excellent\n,An excellent courtroom drama with a unique twi...
1,Andrew Devonshire,10,590,649,18 September 2002,"No bombs, no car chases but edge of the seat ...","This film is superb, in fact as Shakespeare on..."
2,juho69,9,508,565,23 October 2004,Simple but great.\n,'12 Angry Men' is an outstanding film. It is p...
3,Thelightbulb,10,363,405,22 September 2005,If you only ever see one Black and White movi...,"I watched this film for the first time, when i..."
4,Freddy_Levit,10,418,487,11 January 2005,Should be in everyone's top ten list of great...,........Films rarely get this uplifting and br...


In [71]:
# STEP 41: Download raw review files for available Top 100 movies

import requests
import os
import re
from urllib.parse import quote

# Folder to store raw review files
raw_folder = "../01_Raw_Data/MansaT_Top100_Reviews"

os.makedirs(raw_folder, exist_ok=True)

# Get all review files again
api_url = (
    "https://huggingface.co/api/datasets/"
    "MansaT/Movie-Dataset/tree/main/data/"
    "2_reviews_per_movie_raw?recursive=false"
)

response = requests.get(api_url, timeout=30)

print("==============================================")
print("DOWNLOADING TOP 100 RAW REVIEW DATA")
print("==============================================")

if response.status_code != 200:
    print("❌ Could not access review files.")
else:

    files = response.json()

    csv_files = [
        item["path"]
        for item in files
        if item["type"] == "file" and item["path"].endswith(".csv")
    ]

    # Function for matching movie names
    def normalize_title(title):
        title = title.lower()
        title = re.sub(r"[^a-z0-9]+", " ", title)
        title = re.sub(r"\s+", " ", title).strip()
        return title

    # Create normalized Top 100 dictionary
    top100_normalized = {
        normalize_title(movie): movie
        for movie in imdb_top_100
    }

    matched_files = []

    # Find files belonging to our Top 100
    for file_path in csv_files:

        filename = file_path.split("/")[-1]
        movie_name = filename[:-4]

        # Remove year from filename
        movie_without_year = re.sub(
            r"\s+\d{4}$", "", movie_name
        ).strip()

        normalized_movie = normalize_title(movie_without_year)

        if normalized_movie in top100_normalized:
            matched_files.append(
                (file_path, top100_normalized[normalized_movie])
            )

    print("Top 100 movies matched:", len(matched_files))
    print()

    # Download matched files
    successful = 0
    failed = 0

    for i, (file_path, movie_name) in enumerate(
        matched_files, start=1
    ):

        filename = file_path.split("/")[-1]

        download_url = (
            "https://huggingface.co/datasets/"
            "MansaT/Movie-Dataset/resolve/main/"
            + quote(file_path)
        )

        try:

            file_response = requests.get(
                download_url,
                timeout=60
            )

            if file_response.status_code == 200:

                save_path = os.path.join(
                    raw_folder,
                    filename
                )

                with open(
                    save_path,
                    "wb"
                ) as file:

                    file.write(file_response.content)

                successful += 1

                print(
                    f"[{i}/{len(matched_files)}] "
                    f"✓ {movie_name}"
                )

            else:

                failed += 1

                print(
                    f"[{i}/{len(matched_files)}] "
                    f"❌ {movie_name}"
                )

        except Exception as error:

            failed += 1

            print(
                f"[{i}/{len(matched_files)}] "
                f"❌ {movie_name} - {error}"
            )

    print("\n==============================================")
    print("DOWNLOAD COMPLETE")
    print("==============================================")

    print("Files successfully downloaded:", successful)
    print("Files failed:", failed)
    print("Raw data folder:", raw_folder)

DOWNLOADING TOP 100 RAW REVIEW DATA
Top 100 movies matched: 70

[1/70] ✓ 12 Angry Men
[2/70] ✓ 3 Idiots
[3/70] ✓ Alien
[4/70] ✓ Aliens
[5/70] ✓ Amadeus
[6/70] ✓ American Beauty
[7/70] ✓ American History X
[8/70] ✓ Apocalypse Now
[9/70] ✓ Avengers: Endgame
[10/70] ✓ Avengers: Infinity War
[11/70] ✓ Back to the Future
[12/70] ✓ Braveheart
[13/70] ✓ Casablanca
[14/70] ✓ City of God
[15/70] ✓ Coco
[16/70] ✓ Come and See
[17/70] ✓ Django Unchained
[18/70] ✓ Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb
[19/70] ✓ Eternal Sunshine of the Spotless Mind
[20/70] ✓ Fight Club
[21/70] ✓ Forrest Gump
[22/70] ✓ Gladiator
[23/70] ✓ Good Will Hunting
[24/70] ✓ Goodfellas
[25/70] ✓ Grave of the Fireflies
[26/70] ✓ Incendies
[27/70] ✓ Inception
[28/70] ✓ Inglourious Basterds
[29/70] ✓ Interstellar
[30/70] ✓ It's a Wonderful Life
[31/70] ✓ Joker
[32/70] ✓ Life Is Beautiful
[33/70] ✓ Memento
[34/70] ✓ Oldboy
[35/70] ✓ Once Upon a Time in America
[36/70] ✓ One Flew Over the Cuckoo's 

In [72]:
# STEP 42: Check downloaded raw review files

import os

raw_folder = "../01_Raw_Data/MansaT_Top100_Reviews"

files = [
    file
    for file in os.listdir(raw_folder)
    if file.lower().endswith(".csv")
]

print("==============================================")
print("DOWNLOADED RAW DATA CHECK")
print("==============================================")

print("CSV files downloaded:", len(files))

print("\nFirst 20 files:")

for file in sorted(files)[:20]:
    print("-", file)

print("\nLast 20 files:")

for file in sorted(files)[-20:]:
    print("-", file)

DOWNLOADED RAW DATA CHECK
CSV files downloaded: 59

First 20 files:
- 12 Angry Men 1957.csv
- 3 Idiots 2009.csv
- Alien 1979.csv
- Aliens 1986.csv
- Amadeus 1984.csv
- American Beauty 1999.csv
- American History X 1998.csv
- Apocalypse Now 1979.csv
- Back to the Future 1985.csv
- Braveheart 1995.csv
- Casablanca 1942.csv
- City of God 2002.csv
- Coco 2017.csv
- Come and See 1985.csv
- Django Unchained 2012.csv
- Eternal Sunshine of the Spotless Mind 2004.csv
- Fight Club 1999.csv
- Forrest Gump 1994.csv
- Gladiator 2000.csv
- Good Will Hunting 1997.csv

Last 20 files:
- Rear Window 1954.csv
- Requiem for a Dream 2000.csv
- Saving Private Ryan 1998.csv
- Schindler's List 1993.csv
- Se7en 1995.csv
- Singin' in the Rain 1952.csv
- Spirited Away 2001.csv
- The Dark Knight 2008.csv
- The Dark Knight Rises 2012.csv
- The Departed 2006.csv
- The Godfather 1972.csv
- The Great Dictator 1940.csv
- The Green Mile 1999.csv
- The Intouchables 2011.csv
- The Lion King 1994.csv
- The Lion King 2019.

In [73]:
# STEP 43: Load and inspect the combined raw dataset

import pandas as pd
from pathlib import Path

# Path of the saved raw dataset
raw_file = Path("../01_Raw_Data/raw_movie_reviews.csv")

# Load dataset
raw_df = pd.read_csv(raw_file)

print("==============================================")
print("RAW DATASET INSPECTION")
print("==============================================")

print("Dataset shape:", raw_df.shape)

print("\nColumn names:")
for column in raw_df.columns:
    print("-", column)

print("\nFirst 5 records:")
display(raw_df.head())

print("\nData types:")
print(raw_df.dtypes)

RAW DATASET INSPECTION
Dataset shape: (1000, 3)

Column names:
- Movie_Name
- Reviewer
- Review_Text

First 5 records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...



Data types:
Movie_Name     str
Reviewer       str
Review_Text    str
dtype: object


In [74]:
# STEP 43 (CORRECTED): Load the new IMDb Top 100 review data

from pathlib import Path
import pandas as pd

raw_folder = Path("../01_Raw_Data/MansaT_Top100_Reviews")

csv_files = list(raw_folder.glob("*.csv"))

print("==============================================")
print("IMDb TOP 100 RAW DATA CHECK")
print("==============================================")
print("CSV files found:", len(csv_files))

# Read all downloaded movie review files
all_reviews = []

for file in csv_files:
    df = pd.read_csv(file)
    
    # Store movie name from filename
    df["movie_name"] = file.stem
    
    all_reviews.append(df)

# Combine all files
imdb_raw_df = pd.concat(all_reviews, ignore_index=True)

print("\nTotal review records:", len(imdb_raw_df))
print("Total columns:", len(imdb_raw_df.columns))

print("\nColumn names:")
for column in imdb_raw_df.columns:
    print("-", column)

print("\nFirst 5 records:")
display(imdb_raw_df.head())

IMDb TOP 100 RAW DATA CHECK
CSV files found: 59

Total review records: 106362
Total columns: 8

Column names:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

First 5 records:


,username,rating,helpful,total,date,title,review,movie_name
0,vukodlak,10,764,829,1 July 2000,Excellent\n,An excellent courtroom drama with a unique twi...,12 Angry Men 1957
1,Andrew Devonshire,10,590,649,18 September 2002,"No bombs, no car chases but edge of the seat ...","This film is superb, in fact as Shakespeare on...",12 Angry Men 1957
2,juho69,9,508,565,23 October 2004,Simple but great.\n,'12 Angry Men' is an outstanding film. It is p...,12 Angry Men 1957
3,Thelightbulb,10,363,405,22 September 2005,If you only ever see one Black and White movi...,"I watched this film for the first time, when i...",12 Angry Men 1957
4,Freddy_Levit,10,418,487,11 January 2005,Should be in everyone's top ten list of great...,........Films rarely get this uplifting and br...,12 Angry Men 1957


In [75]:
# STEP 44: Check missing values

print("==============================================")
print("MISSING VALUES CHECK")
print("==============================================")

missing_values = imdb_raw_df.isnull().sum()

print("\nMissing values in each column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

MISSING VALUES CHECK

Missing values in each column:
username      0
rating        0
helpful       0
total         0
date          0
title         0
review        0
movie_name    0
dtype: int64

Total missing values: 0


In [76]:
# STEP 45: Check duplicate records

print("=" * 50)
print("DUPLICATE RECORDS CHECK")
print("=" * 50)

duplicate_count = imdb_raw_df.duplicated().sum()

print("\nTotal duplicate records:", duplicate_count)

if duplicate_count == 0:
    print("✓ No duplicate records found.")
else:
    print("⚠ Duplicate records found.")

DUPLICATE RECORDS CHECK

Total duplicate records: 0
✓ No duplicate records found.


In [77]:
# STEP 45: Clean text fields

from bs4 import BeautifulSoup
import re

print("==============================================")
print("TEXT DATA CLEANING")
print("==============================================")

# Create a copy so the original raw data remains unchanged
cleaned_df = imdb_raw_df.copy()

# Function to clean text
def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Remove HTML tags
    text = BeautifulSoup(str(text), "html.parser").get_text()
    
    # Remove extra spaces and line breaks
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading and trailing spaces
    text = text.strip()
    
    return text


# Clean review title
cleaned_df["title"] = cleaned_df["title"].apply(clean_text)

# Clean review text
cleaned_df["review"] = cleaned_df["review"].apply(clean_text)

# Clean username
cleaned_df["username"] = cleaned_df["username"].apply(clean_text)

# Clean movie name
cleaned_df["movie_name"] = cleaned_df["movie_name"].apply(clean_text)


print("Text cleaning completed successfully.")

print("\nSample cleaned records:")
print(cleaned_df[["movie_name", "username", "title", "review"]].head())

TEXT DATA CLEANING


C:\Users\vyshn\AppData\Local\Temp\ipykernel_2732\3948201278.py:19: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(str(text), "html.parser").get_text()


Text cleaning completed successfully.

Sample cleaned records:
          movie_name           username  \
0  12 Angry Men 1957           vukodlak   
1  12 Angry Men 1957  Andrew Devonshire   
2  12 Angry Men 1957             juho69   
3  12 Angry Men 1957       Thelightbulb   
4  12 Angry Men 1957       Freddy_Levit   

                                               title  \
0                                          Excellent   
1  No bombs, no car chases but edge of the seat s...   
2                                  Simple but great.   
3  If you only ever see one Black and White movie...   
4  Should be in everyone's top ten list of greate...   

                                              review  
0  An excellent courtroom drama with a unique twi...  
1  This film is superb, in fact as Shakespeare on...  
2  '12 Angry Men' is an outstanding film. It is p...  
3  I watched this film for the first time, when i...  
4  ........Films rarely get this uplifting and br...  


In [78]:
# STEP 46: Convert numeric columns to proper numeric format

print("==============================================")
print("NUMERIC DATA CONVERSION")
print("==============================================")

# Convert rating to numeric
cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"],
    errors="coerce"
)

# Convert helpful votes to numeric
cleaned_df["helpful"] = pd.to_numeric(
    cleaned_df["helpful"],
    errors="coerce"
)

# Convert total votes to numeric
cleaned_df["total"] = pd.to_numeric(
    cleaned_df["total"],
    errors="coerce"
)

print("Numeric conversion completed successfully.")

print("\nData types after conversion:")
print(cleaned_df[["rating", "helpful", "total"]].dtypes)

print("\nSample numeric data:")
print(cleaned_df[["username", "rating", "helpful", "total"]].head())

NUMERIC DATA CONVERSION
Numeric conversion completed successfully.

Data types after conversion:
rating     float64
helpful      int64
total        int64
dtype: object

Sample numeric data:
            username  rating  helpful  total
0           vukodlak    10.0      764    829
1  Andrew Devonshire    10.0      590    649
2             juho69     9.0      508    565
3       Thelightbulb    10.0      363    405
4       Freddy_Levit    10.0      418    487


In [79]:
# STEP 47: Check numeric columns for invalid or missing values

print("==============================================")
print("NUMERIC DATA QUALITY CHECK")
print("==============================================")

numeric_columns = ["rating", "helpful", "total"]

print("\nMissing values in numeric columns:")
print(cleaned_df[numeric_columns].isnull().sum())

print("\nInvalid negative values:")

for column in numeric_columns:
    negative_count = (cleaned_df[column] < 0).sum()
    print(column, ":", negative_count)

print("\nRating range:")
print("Minimum rating:", cleaned_df["rating"].min())
print("Maximum rating:", cleaned_df["rating"].max())

NUMERIC DATA QUALITY CHECK

Missing values in numeric columns:
rating     10599
helpful        0
total          0
dtype: int64

Invalid negative values:
rating : 0
helpful : 0
total : 0

Rating range:
Minimum rating: 1.0
Maximum rating: 10.0


In [80]:
# STEP 48: Remove records with missing ratings

print("==============================================")
print("REMOVING MISSING RATINGS")
print("==============================================")

before_count = len(cleaned_df)

# Remove records where rating is missing
cleaned_df = cleaned_df.dropna(subset=["rating"]).copy()

after_count = len(cleaned_df)

removed_count = before_count - after_count

print("Records before removing missing ratings:", before_count)
print("Records removed:", removed_count)
print("Records after removing missing ratings:", after_count)

print("\nMissing ratings remaining:",
      cleaned_df["rating"].isnull().sum())

REMOVING MISSING RATINGS
Records before removing missing ratings: 106362
Records removed: 10599
Records after removing missing ratings: 95763

Missing ratings remaining: 0


In [81]:
# STEP 49: Final cleaned dataset inspection

print("==============================================")
print("FINAL CLEANED DATASET INSPECTION")
print("==============================================")

print("\nDataset shape:")
print(cleaned_df.shape)

print("\nColumn names:")
for column in cleaned_df.columns:
    print("-", column)

print("\nData types:")
print(cleaned_df.dtypes)

print("\nFirst 5 cleaned records:")
display(cleaned_df.head())

FINAL CLEANED DATASET INSPECTION

Dataset shape:
(95763, 8)

Column names:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Data types:
username          str
rating        float64
helpful         int64
total           int64
date              str
title             str
review            str
movie_name        str
dtype: object

First 5 cleaned records:


,username,rating,helpful,total,date,title,review,movie_name
0,vukodlak,10.0,764,829,1 July 2000,Excellent,An excellent courtroom drama with a unique twi...,12 Angry Men 1957
1,Andrew Devonshire,10.0,590,649,18 September 2002,"No bombs, no car chases but edge of the seat s...","This film is superb, in fact as Shakespeare on...",12 Angry Men 1957
2,juho69,9.0,508,565,23 October 2004,Simple but great.,'12 Angry Men' is an outstanding film. It is p...,12 Angry Men 1957
3,Thelightbulb,10.0,363,405,22 September 2005,If you only ever see one Black and White movie...,"I watched this film for the first time, when i...",12 Angry Men 1957
4,Freddy_Levit,10.0,418,487,11 January 2005,Should be in everyone's top ten list of greate...,........Films rarely get this uplifting and br...,12 Angry Men 1957


In [82]:
# STEP 50: Final duplicate check

print("==============================================")
print("FINAL DUPLICATE CHECK")
print("==============================================")

duplicate_count = cleaned_df.duplicated().sum()

print("\nTotal duplicate records:", duplicate_count)

if duplicate_count == 0:
    print("✓ Final dataset contains no duplicate records.")
else:
    print("⚠ Duplicate records found.")

FINAL DUPLICATE CHECK

Total duplicate records: 3
⚠ Duplicate records found.


In [83]:
# STEP 51: Remove duplicate records

print("==============================================")
print("REMOVING DUPLICATE RECORDS")
print("==============================================")

before_count = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

after_count = len(cleaned_df)

removed_count = before_count - after_count

print("\nRecords before removing duplicates:", before_count)
print("Duplicate records removed:", removed_count)
print("Records after removing duplicates:", after_count)

print("\nDuplicates remaining:", cleaned_df.duplicated().sum())

REMOVING DUPLICATE RECORDS

Records before removing duplicates: 95763
Duplicate records removed: 3
Records after removing duplicates: 95760

Duplicates remaining: 0


In [84]:
# STEP 52: Final missing-value check

print("==============================================")
print("FINAL MISSING VALUE CHECK")
print("==============================================")

missing_values = cleaned_df.isnull().sum()

print("\nMissing values in each column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

if missing_values.sum() == 0:
    print("✓ Final dataset contains no missing values.")
else:
    print("⚠ Missing values found.")

FINAL MISSING VALUE CHECK

Missing values in each column:
username      0
rating        0
helpful       0
total         0
date          0
title         0
review        0
movie_name    0
dtype: int64

Total missing values: 0
✓ Final dataset contains no missing values.


In [85]:
# STEP 49: Final cleaned dataset inspection

print("=" * 55)
print("FINAL CLEANED DATASET INSPECTION")
print("=" * 55)

print("\nDataset shape:")
print(cleaned_df.shape)

print("\nColumn names:")
for column in cleaned_df.columns:
    print("-", column)

print("\nData types:")
print(cleaned_df.dtypes)

print("\nFirst 5 cleaned records:")
display(cleaned_df.head())

print("\nFinal missing values:")
print(cleaned_df.isnull().sum())

print("\nTotal missing values:", cleaned_df.isnull().sum().sum())

print("\nFinal duplicate records:", cleaned_df.duplicated().sum())

FINAL CLEANED DATASET INSPECTION

Dataset shape:
(95760, 8)

Column names:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Data types:
username          str
rating        float64
helpful         int64
total           int64
date              str
title             str
review            str
movie_name        str
dtype: object

First 5 cleaned records:


,username,rating,helpful,total,date,title,review,movie_name
0,vukodlak,10.0,764,829,1 July 2000,Excellent,An excellent courtroom drama with a unique twi...,12 Angry Men 1957
1,Andrew Devonshire,10.0,590,649,18 September 2002,"No bombs, no car chases but edge of the seat s...","This film is superb, in fact as Shakespeare on...",12 Angry Men 1957
2,juho69,9.0,508,565,23 October 2004,Simple but great.,'12 Angry Men' is an outstanding film. It is p...,12 Angry Men 1957
3,Thelightbulb,10.0,363,405,22 September 2005,If you only ever see one Black and White movie...,"I watched this film for the first time, when i...",12 Angry Men 1957
4,Freddy_Levit,10.0,418,487,11 January 2005,Should be in everyone's top ten list of greate...,........Films rarely get this uplifting and br...,12 Angry Men 1957



Final missing values:
username      0
rating        0
helpful       0
total         0
date          0
title         0
review        0
movie_name    0
dtype: int64

Total missing values: 0

Final duplicate records: 0


In [86]:
# STEP 54: Save the final cleaned dataset

import os

# Folder for cleaned data
cleaned_folder = r"C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1 - Movie review Analysis\02_Cleaned_Data"

# Create folder if it does not exist
os.makedirs(cleaned_folder, exist_ok=True)

# File path
cleaned_file = os.path.join(
    cleaned_folder,
    "PPS1_Movie_Reviews_Cleaned.csv"
)

# Save cleaned dataset
cleaned_df.to_csv(cleaned_file, index=False, encoding="utf-8")

print("==============================================")
print("FINAL CLEANED DATASET SAVED")
print("==============================================")

print("File name:", "PPS1_Movie_Reviews_Cleaned.csv")
print("Location:", cleaned_file)
print("Records saved:", len(cleaned_df))
print("Columns saved:", len(cleaned_df.columns))

FINAL CLEANED DATASET SAVED
File name: PPS1_Movie_Reviews_Cleaned.csv
Location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1 - Movie review Analysis\02_Cleaned_Data\PPS1_Movie_Reviews_Cleaned.csv
Records saved: 95760
Columns saved: 8


In [87]:
# STEP 55: Verify the saved cleaned dataset

import pandas as pd
import os

# Check whether the file exists
file_exists = os.path.exists(cleaned_file)

print("==============================================")
print("SAVED DATASET VERIFICATION")
print("==============================================")

print("File exists:", file_exists)

if file_exists:
    # Read the saved CSV
    saved_df = pd.read_csv(cleaned_file)

    print("Records in saved file:", len(saved_df))
    print("Columns in saved file:", len(saved_df.columns))

    print("\nColumn names:")
    for column in saved_df.columns:
        print("-", column)

    print("\nSaved dataset shape:", saved_df.shape)

    print("\nFirst 5 records:")
    display(saved_df.head())
else:
    print("❌ File was not found.")

SAVED DATASET VERIFICATION
File exists: True
Records in saved file: 95760
Columns in saved file: 8

Column names:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Saved dataset shape: (95760, 8)

First 5 records:


,username,rating,helpful,total,date,title,review,movie_name
0,vukodlak,10.0,764,829,1 July 2000,Excellent,An excellent courtroom drama with a unique twi...,12 Angry Men 1957
1,Andrew Devonshire,10.0,590,649,18 September 2002,"No bombs, no car chases but edge of the seat s...","This film is superb, in fact as Shakespeare on...",12 Angry Men 1957
2,juho69,9.0,508,565,23 October 2004,Simple but great.,'12 Angry Men' is an outstanding film. It is p...,12 Angry Men 1957
3,Thelightbulb,10.0,363,405,22 September 2005,If you only ever see one Black and White movie...,"I watched this film for the first time, when i...",12 Angry Men 1957
4,Freddy_Levit,10.0,418,487,11 January 2005,Should be in everyone's top ten list of greate...,........Films rarely get this uplifting and br...,12 Angry Men 1957


In [88]:
# STEP 56: Final dataset statistics

print("==============================================")
print("FINAL DATASET STATISTICS")
print("==============================================")

print("\nTotal records:", len(saved_df))
print("Total columns:", len(saved_df.columns))

print("\nRating statistics:")
print(saved_df["rating"].describe())

print("\nHelpful votes statistics:")
print(saved_df["helpful"].describe())

print("\nTotal votes statistics:")
print(saved_df["total"].describe())

FINAL DATASET STATISTICS

Total records: 95760
Total columns: 8

Rating statistics:
count    95760.000000
mean         8.343567
std          2.495968
min          1.000000
25%          8.000000
50%          9.000000
75%         10.000000
max         10.000000
Name: rating, dtype: float64

Helpful votes statistics:
count    95760.000000
mean         7.166374
std         58.688959
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max       7092.000000
Name: helpful, dtype: float64

Total votes statistics:
count    95760.000000
mean        14.057362
std         84.688319
min          0.000000
25%          1.000000
50%          3.000000
75%          7.000000
max       8206.000000
Name: total, dtype: float64


In [89]:
# STEP 57: Check movie distribution

print("==============================================")
print("MOVIE DISTRIBUTION CHECK")
print("==============================================")

# Count unique movies
unique_movies = saved_df["movie_name"].nunique()

print("\nTotal unique movies:", unique_movies)

# Count reviews for each movie
movie_review_counts = saved_df["movie_name"].value_counts()

print("\nReviews per movie:")
print(movie_review_counts)

print("\nTop 10 movies by number of reviews:")
print(movie_review_counts.head(10))

MOVIE DISTRIBUTION CHECK

Total unique movies: 59

Reviews per movie:
movie_name
Joker 2019                                    9935
The Dark Knight 2008                          6493
Interstellar 2014                             3791
Inception 2010                                3556
The Godfather 1972                            3448
The Matrix 1999                               3381
Fight Club 1999                               3085
The Dark Knight Rises 2012                    2861
The Lion King 2019                            2814
Pulp Fiction 1994                             2600
Parasite 2019                                 2347
American Beauty 1999                          2278
The Departed 2006                             2189
Forrest Gump 1994                             2080
Gladiator 2000                                2023
Saving Private Ryan 1998                      1988
Memento 2000                                  1869
Requiem for a Dream 2000                      1745
I

In [90]:
# STEP 58: Rating distribution

print("==============================================")
print("RATING DISTRIBUTION")
print("==============================================")

rating_counts = saved_df["rating"].value_counts().sort_index()

print("\nNumber of reviews for each rating:")
print(rating_counts)

print("\nTotal ratings counted:", rating_counts.sum())

RATING DISTRIBUTION

Number of reviews for each rating:
rating
1.0      4601
2.0      1607
3.0      1789
4.0      1913
5.0      2638
6.0      3500
7.0      5428
8.0      9784
9.0     17312
10.0    47188
Name: count, dtype: int64

Total ratings counted: 95760


In [91]:
# STEP 56: FINAL DATASET SUMMARY

print("=" * 55)
print("FINAL DATASET SUMMARY")
print("=" * 55)

print("\nTotal number of reviews:", len(saved_df))
print("Total number of movies:", saved_df["movie_name"].nunique())

print("\nDataset columns:")
for column in saved_df.columns:
    print("-", column)

print("\nDate range:")
print("First review date:", saved_df["date"].min())
print("Last review date:", saved_df["date"].max())

print("\nRating range:")
print("Minimum rating:", saved_df["rating"].min())
print("Maximum rating:", saved_df["rating"].max())
print("Average rating:", round(saved_df["rating"].mean(), 2))

print("\nData quality:")
print("Missing values:", saved_df.isnull().sum().sum())
print("Duplicate records:", saved_df.duplicated().sum())

print("\n" + "=" * 55)
print("DATASET PROCESSING COMPLETED")
print("=" * 55)

FINAL DATASET SUMMARY

Total number of reviews: 95760
Total number of movies: 59

Dataset columns:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Date range:
First review date: 1 April 1999
Last review date: 9 September 2019

Rating range:
Minimum rating: 1.0
Maximum rating: 10.0
Average rating: 8.34

Data quality:
Missing values: 2
Duplicate records: 0

DATASET PROCESSING COMPLETED


In [93]:
# STEP 57: UPDATE SOURCE URL DOCUMENTATION - FIXED

from pathlib import Path

source_file = Path("../07_Source_URLs/source_urls.txt")

source_content = """PPS 1 - Movie Review Analysis
Dataset Source Information
==========================================

Primary Data Source:
https://github.com/SimpliSolve/RecSysData

IMDb Top Movies Source:
https://github.com/SimpliSolve/RecSysData/tree/master/top250

IMDb User Review Dataset:
https://github.com/SimpliSolve/RecSysData/tree/master/user_review_data_daily_search

Top 250 Movie Data:
https://raw.githubusercontent.com/SimpliSolve/RecSysData/master/top250/top250.csv

Original IMDb Reference:
https://www.imdb.com/chart/top/

Tools Used:
- Python
- Requests
- Pandas
- BeautifulSoup

Dataset Fields:
1. username
2. rating
3. helpful
4. total
5. date
6. title
7. review
8. movie_name

Data Processing Performed:
- Missing value checking
- Missing rating removal
- Duplicate record detection
- Duplicate record removal
- Data type conversion
- Text cleaning
- Rating validation
- Final dataset verification

Final Dataset:
95,760 review records
8 columns
0 missing values
0 duplicate records

Note:
The final review data used in this project comes from a publicly
available pre-collected research dataset originating from IMDb.
Direct live scraping of IMDb was not used.

Purpose:
Dataset Collection, Cleaning and Processing
"""

source_file.write_text(source_content, encoding="utf-8")

print("=" * 55)
print("SOURCE URL FILE UPDATED")
print("=" * 55)

print("\nFile:", source_file.resolve())
print("Status: Successfully updated")
print("\nFinal source documentation is ready.")

SOURCE URL FILE UPDATED

File: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\07_Source_URLs\source_urls.txt
Status: Successfully updated

Final source documentation is ready.


In [94]:
# STEP 58: VERIFY SOURCE URL DOCUMENTATION

from pathlib import Path

source_file = Path("../07_Source_URLs/source_urls.txt")

print("=" * 55)
print("SOURCE URL FILE VERIFICATION")
print("=" * 55)

print("\nFile exists:", source_file.exists())

if source_file.exists():
    print("File location:", source_file.resolve())

    content = source_file.read_text(encoding="utf-8")

    print("\nFile contents:")
    print("-" * 55)
    print(content)
    print("-" * 55)

    print("\n✓ Source URL documentation verified successfully.")
else:
    print("\n✗ Source URL file was not found.")

SOURCE URL FILE VERIFICATION

File exists: True
File location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\07_Source_URLs\source_urls.txt

File contents:
-------------------------------------------------------
PPS 1 - Movie Review Analysis
Dataset Source Information

Primary Data Source:
https://github.com/SimpliSolve/RecSysData

IMDb Top Movies Source:
https://github.com/SimpliSolve/RecSysData/tree/master/top250

IMDb User Review Dataset:
https://github.com/SimpliSolve/RecSysData/tree/master/user_review_data_daily_search

Top 250 Movie Data:
https://raw.githubusercontent.com/SimpliSolve/RecSysData/master/top250/top250.csv

Original IMDb Reference:
https://www.imdb.com/chart/top/

Tools Used:
- Python
- Requests
- Pandas
- BeautifulSoup

Dataset Fields:
1. username
2. rating
3. helpful
4. total
5. date
6. title
7. review
8. movie_name

Data Processing Performed:
- Missing value checking
- Missing rating removal
- Duplicate record detection
- Duplic

In [96]:
# STEP 58: Create Project Methodology Documentation

from pathlib import Path

documentation_folder = Path(
    r"C:\Users\Miniconda3\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1 - Movie Review Analysis\05_Documentation"
)

documentation_folder.mkdir(parents=True, exist_ok=True)

methodology = """
MOVIE REVIEW ANALYSIS
PPS 1 - Dataset Collection, Cleaning and Processing

1. PROJECT OBJECTIVE
--------------------
The objective of this project is to collect, clean and process movie
review data for analysis using Python and Pandas.

The dataset contains movie review information along with reviewer
details, ratings, review dates and helpfulness information.

2. DATA SOURCE
--------------
The final dataset was obtained from a publicly available pre-collected
research dataset originating from IMDb.

Source Repository:
https://github.com/SimpliSolve/RecSysData

The project did not perform direct live scraping of IMDb.
The publicly available pre-collected dataset was used for academic
and educational purposes.

3. TOOLS AND TECHNOLOGIES USED
------------------------------
Python
Pandas
Requests
BeautifulSoup
Jupyter Notebook

4. DATA COLLECTION
------------------
The publicly available research dataset was accessed using Python.
The required movie review records were collected and combined into
a single Pandas DataFrame.

The dataset contains fields such as:

- Username
- Rating
- Helpful votes
- Total votes
- Review date
- Review title
- Review text
- Movie name

5. DATA CLEANING
----------------
The collected data was checked for data quality problems.

The following cleaning operations were performed:

- Missing values were checked.
- Duplicate records were identified and removed.
- Unwanted spaces and formatting were cleaned.
- Rating values were converted into numeric format.
- Data types were checked for consistency.
- Invalid or incomplete records were examined.

After cleaning, the final dataset contained:

95760 review records
8 columns
0 missing values
0 duplicate records

6. DATA PREPROCESSING
---------------------
The cleaned dataset was prepared for further analysis.

The rating, helpful votes and total votes columns were converted
to appropriate numeric data types.

Review dates were checked and processed as date-related information.

Text fields such as review titles and review text were retained
for further analysis.

7. DATASET CREATION
-------------------
After completing the cleaning and preprocessing operations, the
processed dataset was saved as a CSV file.

The saved dataset was verified by loading it again and checking:

- File existence
- Number of records
- Number of columns
- Column names
- Missing values
- Duplicate records
- Sample records

8. FINAL DATASET
----------------
Final number of records: 95760
Number of columns: 8
Missing values: 0
Duplicate records: 0

9. CONCLUSION
-------------
The movie review dataset was successfully collected from a publicly
available pre-collected research dataset, cleaned and processed
using Python and Pandas.

The final dataset is ready for further movie review analysis and
visualization.
"""

documentation_file = documentation_folder / "Methodology_Report.txt"

documentation_file.write_text(
    methodology,
    encoding="utf-8"
)

print("=" * 55)
print("STEP 58 - DOCUMENTATION CREATED")
print("=" * 55)

print("\nFile:", documentation_file)
print("Status: Successfully created")
print("\nThe methodology report is ready.")

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\Miniconda3'

In [97]:
# STEP 58: Create Project Methodology Documentation

from pathlib import Path

# Get the project folder automatically from the notebook location
project_folder = Path.cwd().parent

documentation_folder = project_folder / "05_Documentation"

documentation_folder.mkdir(parents=True, exist_ok=True)

methodology = """
MOVIE REVIEW ANALYSIS
PPS 1 - Dataset Collection, Cleaning and Processing

1. PROJECT OBJECTIVE
--------------------
The objective of this project is to collect, clean and process movie
review data for analysis using Python and Pandas.

The dataset contains movie review information along with reviewer
details, ratings, review dates and helpfulness information.

2. DATA SOURCE
--------------
The final dataset was obtained from a publicly available pre-collected
research dataset originating from IMDb.

Source Repository:
https://github.com/SimpliSolve/RecSysData

The project did not perform direct live scraping of IMDb.
The publicly available pre-collected dataset was used for academic
and educational purposes.

3. TOOLS AND TECHNOLOGIES USED
------------------------------
Python
Pandas
Requests
BeautifulSoup
Jupyter Notebook

4. DATA COLLECTION
------------------
The publicly available research dataset was accessed using Python.
The required movie review records were collected and combined into
a single Pandas DataFrame.

The dataset contains fields such as:

- Username
- Rating
- Helpful votes
- Total votes
- Review date
- Review title
- Review text
- Movie name

5. DATA CLEANING
----------------
The collected data was checked for data quality problems.

The following cleaning operations were performed:

- Missing values were checked.
- Duplicate records were identified and removed.
- Unwanted spaces and formatting were cleaned.
- Rating values were converted into numeric format.
- Data types were checked for consistency.
- Invalid or incomplete records were examined.

After cleaning, the final dataset contained:

95760 review records
8 columns
0 missing values
0 duplicate records

6. DATA PREPROCESSING
---------------------
The cleaned dataset was prepared for further analysis.

The rating, helpful votes and total votes columns were converted
to appropriate numeric data types.

Review dates were checked and processed as date-related information.

Text fields such as review titles and review text were retained
for further analysis.

7. DATASET CREATION
-------------------
After completing the cleaning and preprocessing operations, the
processed dataset was saved as a CSV file.

The saved dataset was verified by loading it again and checking:

- File existence
- Number of records
- Number of columns
- Column names
- Missing values
- Duplicate records
- Sample records

8. FINAL DATASET
----------------
Final number of records: 95760
Number of columns: 8
Missing values: 0
Duplicate records: 0

9. CONCLUSION
-------------
The movie review dataset was successfully collected from a publicly
available pre-collected research dataset, cleaned and processed
using Python and Pandas.

The final dataset is ready for further movie review analysis and
visualization.
"""

documentation_file = documentation_folder / "Methodology_Report.txt"

documentation_file.write_text(
    methodology,
    encoding="utf-8"
)

print("=" * 55)
print("STEP 58 - DOCUMENTATION CREATED")
print("=" * 55)

print("\nProject folder:")
print(project_folder)

print("\nDocumentation file:")
print(documentation_file)

print("\nStatus: Successfully created")
print("\nThe methodology report is ready.")

STEP 58 - DOCUMENTATION CREATED

Project folder:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis

Documentation file:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\05_Documentation\Methodology_Report.txt

Status: Successfully created

The methodology report is ready.


In [99]:
# STEP 59: Final Project Summary Check

from pathlib import Path

project_folder = Path.cwd().parent

print("=" * 60)
print("FINAL PROJECT SUMMARY")
print("=" * 60)

print("\nProject:")
print("PPS 1 - Movie Review Analysis")

print("\nDataset Summary:")
print("Records:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("Missing values:", cleaned_df.isnull().sum().sum())
print("Duplicate records:", cleaned_df.duplicated().sum())

print("\nColumns:")
for column in cleaned_df.columns:
    print("-", column)

print("\nProject Folders:")

folders = [
    "01_Raw_Data",
    "02_Cleaned_Data",
    "03_Code",
    "Note_book",
    "05_Documentation",
    "06_Output",
    "07_Source_URLs"
]

for folder in folders:
    folder_path = project_folder / folder
    
    if folder_path.exists():
        print("✓", folder)
    else:
        print("✗", folder)

print("\n" + "=" * 60)
print("FINAL CHECK COMPLETED")
print("=" * 60)

FINAL PROJECT SUMMARY

Project:
PPS 1 - Movie Review Analysis

Dataset Summary:
Records: 95760
Columns: 8
Missing values: 0
Duplicate records: 0

Columns:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Project Folders:
✓ 01_Raw_Data
✓ 02_Cleaned_Data
✗ 03_Code
✓ Note_book
✓ 05_Documentation
✓ 06_Output
✓ 07_Source_URLs

FINAL CHECK COMPLETED


In [101]:
# STEP 60: FINAL PROJECT VERIFICATION

from pathlib import Path

print("=" * 60)
print("STEP 60 - FINAL PROJECT VERIFICATION")
print("=" * 60)

# 1. Check project folders
print("\nPROJECT FOLDER CHECK")
print("-" * 60)

folders_to_check = [
    "01_Raw_Data",
    "02_Cleaned_Data",
    "03_Code",
    "Note_book",
    "05_Documentation",
    "06_Output",
    "07_Source_URLs"
]

for folder_name in folders_to_check:
    folder_path = project_folder / folder_name

    if folder_path.exists():
        print("✓", folder_name, "- folder exists")
    else:
        print("✗", folder_name, "- folder missing")


# 2. Check documentation file
print("\nDOCUMENTATION CHECK")
print("-" * 60)

documentation_file = (
    project_folder /
    "05_Documentation" /
    "Methodology_Report.txt"
)

if documentation_file.exists():
    print("✓ Methodology_Report.txt exists")
else:
    print("✗ Methodology_Report.txt missing")


# 3. Dataset quality check
print("\nDATASET QUALITY CHECK")
print("-" * 60)

print("Total records:", len(cleaned_df))
print("Total columns:", len(cleaned_df.columns))
print("Missing values:", int(cleaned_df.isnull().sum().sum()))
print("Duplicate records:", int(cleaned_df.duplicated().sum()))


# 4. Final status
print("\n" + "=" * 60)
print("FINAL VERIFICATION COMPLETED")
print("=" * 60)

STEP 60 - FINAL PROJECT VERIFICATION

PROJECT FOLDER CHECK
------------------------------------------------------------
✓ 01_Raw_Data - folder exists
✓ 02_Cleaned_Data - folder exists
✗ 03_Code - folder missing
✓ Note_book - folder exists
✓ 05_Documentation - folder exists
✓ 06_Output - folder exists
✓ 07_Source_URLs - folder exists

DOCUMENTATION CHECK
------------------------------------------------------------
✓ Methodology_Report.txt exists

DATASET QUALITY CHECK
------------------------------------------------------------
Total records: 95760
Total columns: 8
Missing values: 0
Duplicate records: 0

FINAL VERIFICATION COMPLETED


In [102]:
# STEP 61: CREATE MISSING CODE FOLDER

from pathlib import Path

code_folder = project_folder / "03_Code"

# Create the folder if it does not exist
code_folder.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("STEP 61 - CODE FOLDER CREATED")
print("=" * 60)

if code_folder.exists():
    print("✓ 03_Code folder exists")
    print("Location:", code_folder)
else:
    print("✗ 03_Code folder could not be created")

STEP 61 - CODE FOLDER CREATED
✓ 03_Code folder exists
Location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\03_Code


In [103]:
# STEP 62: FINAL FOLDER STRUCTURE CHECK

print("=" * 60)
print("STEP 62 - FINAL PROJECT STRUCTURE")
print("=" * 60)

folders_to_check = [
    "01_Raw_Data",
    "02_Cleaned_Data",
    "03_Code",
    "Note_book",
    "05_Documentation",
    "06_Output",
    "07_Source_URLs"
]

all_folders_exist = True

for folder_name in folders_to_check:
    folder_path = project_folder / folder_name

    if folder_path.exists():
        print("✓", folder_name)
    else:
        print("✗", folder_name, "- MISSING")
        all_folders_exist = False

print("\n" + "-" * 60)

if all_folders_exist:
    print("✓ ALL PROJECT FOLDERS ARE READY")
else:
    print("✗ SOME FOLDERS ARE MISSING")

print("=" * 60)

STEP 62 - FINAL PROJECT STRUCTURE
✓ 01_Raw_Data
✓ 02_Cleaned_Data
✓ 03_Code
✓ Note_book
✓ 05_Documentation
✓ 06_Output
✓ 07_Source_URLs

------------------------------------------------------------
✓ ALL PROJECT FOLDERS ARE READY


In [104]:
# STEP 63: SAVE FINAL CLEANED DATASET

cleaned_dataset_file = (
    project_folder /
    "02_Cleaned_Data" /
    "final_cleaned_movie_reviews.csv"
)

cleaned_df.to_csv(
    cleaned_dataset_file,
    index=False,
    encoding="utf-8"
)

print("=" * 60)
print("STEP 63 - FINAL DATASET SAVED")
print("=" * 60)

print("File:", cleaned_dataset_file)
print("Records:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("Status: Successfully saved")

STEP 63 - FINAL DATASET SAVED
File: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\final_cleaned_movie_reviews.csv
Records: 95760
Columns: 8
Status: Successfully saved


In [105]:
# STEP 64: VERIFY FINAL CLEANED DATASET

import pandas as pd

# Load the saved CSV
verified_df = pd.read_csv(cleaned_dataset_file)

print("=" * 60)
print("STEP 64 - FINAL DATASET VERIFICATION")
print("=" * 60)

print("File exists:", cleaned_dataset_file.exists())
print("Records:", len(verified_df))
print("Columns:", len(verified_df.columns))

print("\nColumn names:")
for column in verified_df.columns:
    print("-", column)

print("\nMissing values:", int(verified_df.isnull().sum().sum()))
print("Duplicate records:", int(verified_df.duplicated().sum()))

print("\nFirst 5 records:")
display(verified_df.head())

print("\n" + "=" * 60)
print("FINAL CSV VERIFICATION COMPLETED")
print("=" * 60)

STEP 64 - FINAL DATASET VERIFICATION
File exists: True
Records: 95760
Columns: 8

Column names:
- username
- rating
- helpful
- total
- date
- title
- review
- movie_name

Missing values: 2
Duplicate records: 0

First 5 records:


,username,rating,helpful,total,date,title,review,movie_name
0,vukodlak,10.0,764,829,1 July 2000,Excellent,An excellent courtroom drama with a unique twi...,12 Angry Men 1957
1,Andrew Devonshire,10.0,590,649,18 September 2002,"No bombs, no car chases but edge of the seat s...","This film is superb, in fact as Shakespeare on...",12 Angry Men 1957
2,juho69,9.0,508,565,23 October 2004,Simple but great.,'12 Angry Men' is an outstanding film. It is p...,12 Angry Men 1957
3,Thelightbulb,10.0,363,405,22 September 2005,If you only ever see one Black and White movie...,"I watched this film for the first time, when i...",12 Angry Men 1957
4,Freddy_Levit,10.0,418,487,11 January 2005,Should be in everyone's top ten list of greate...,........Films rarely get this uplifting and br...,12 Angry Men 1957



FINAL CSV VERIFICATION COMPLETED


In [106]:
# STEP 65: CREATE PYTHON CODE FILE

code_file = project_folder / "03_Code" / "pps1_movie_review_analysis.py"

# Collect all executed Python code from this notebook session
code_content = "\n\n".join(
    [code for code in In[1:] if code.strip()]
)

# Save the code
code_file.write_text(code_content, encoding="utf-8")

print("=" * 60)
print("STEP 65 - PYTHON CODE FILE CREATED")
print("=" * 60)

print("File:", code_file)
print("File exists:", code_file.exists())
print("Status: Successfully created")

STEP 65 - PYTHON CODE FILE CREATED
File: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\03_Code\pps1_movie_review_analysis.py
File exists: True
Status: Successfully created


In [108]:
# STEP 64 - FINAL SUBMISSION FILE CHECK

from pathlib import Path

project_folder = Path.cwd().parent

print("=" * 65)
print("STEP 64 - FINAL SUBMISSION FILE CHECK")
print("=" * 65)

# Check project folders
items_to_check = {
    "01_Raw_Data": project_folder / "01_Raw_Data",
    "02_Cleaned_Data": project_folder / "02_Cleaned_Data",
    "03_Code": project_folder / "03_Code",
    "Note_book": project_folder / "Note_book",
    "05_Documentation": project_folder / "05_Documentation",
    "06_Output": project_folder / "06_Output",
    "07_Source_URLs": project_folder / "07_Source_URLs",
}

print("\nPROJECT COMPONENTS")
print("-" * 65)

for name, path in items_to_check.items():
    if path.exists():
        print("✓", name, "- READY")
    else:
        print("✗", name, "- MISSING")

# Check important files
print("\n" + "=" * 65)
print("IMPORTANT FILES")
print("=" * 65)

important_extensions = [".csv", ".xlsx", ".txt", ".ipynb"]

for extension in important_extensions:
    files_found = list(project_folder.rglob(f"*{extension}"))

    print(f"\n{extension.upper()} files found: {len(files_found)}")

    for file in files_found[:15]:
        print(" -", file.relative_to(project_folder))

# Dataset summary
print("\n" + "=" * 65)
print("DATASET SUMMARY")
print("=" * 65)

print("Final records:", len(cleaned_df))
print("Final columns:", len(cleaned_df.columns))
print("Missing values:", cleaned_df.isnull().sum().sum())
print("Duplicate records:", cleaned_df.duplicated().sum())

print("\n" + "=" * 65)
print("STEP 64 COMPLETED")
print("=" * 65)

STEP 64 - FINAL SUBMISSION FILE CHECK

PROJECT COMPONENTS
-----------------------------------------------------------------
✓ 01_Raw_Data - READY
✓ 02_Cleaned_Data - READY
✓ 03_Code - READY
✓ Note_book - READY
✓ 05_Documentation - READY
✓ 06_Output - READY
✓ 07_Source_URLs - READY

IMPORTANT FILES

.CSV files found: 66
 - 01_Raw_Data\raw_imdb_top100_reviews.csv
 - 01_Raw_Data\raw_movie_reviews.csv
 - 02_Cleaned_Data\cleaned_movie_reviews.csv
 - 02_Cleaned_Data\final_cleaned_movie_reviews.csv
 - 02_Cleaned_Data\processed_movie_reviews.csv
 - 06_Output\final_movie_reviews.csv
 - Raw_Data\raw_movie_reviews.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\12 Angry Men 1957.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\3 Idiots 2009.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\Alien 1979.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\Aliens 1986.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\Amadeus 1984.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\American Beauty 1999.csv
 - 01_Raw_Data\MansaT_Top100_Reviews\American Hist

In [109]:
# STEP 65 - CHECK FINAL CLEANED DATASET

from pathlib import Path

cleaned_folder = project_folder / "02_Cleaned_Data"

print("=" * 65)
print("STEP 65 - FINAL CLEANED DATASET CHECK")
print("=" * 65)

print("\nCleaned Data Folder:")
print(cleaned_folder)

csv_files = list(cleaned_folder.glob("*.csv"))

print("\nCSV files found:", len(csv_files))

if len(csv_files) > 0:
    print("\nCleaned dataset files:")
    for file in csv_files:
        print("✓", file.name)
        print("  Size:", round(file.stat().st_size / (1024 * 1024), 2), "MB")
        
    print("\n✓ FINAL CLEANED DATASET EXISTS")
else:
    print("\n✗ No cleaned CSV file found")

print("\n" + "=" * 65)
print("STEP 65 COMPLETED")
print("=" * 65)


STEP 65 - FINAL CLEANED DATASET CHECK

Cleaned Data Folder:
C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data

CSV files found: 3

Cleaned dataset files:
✓ cleaned_movie_reviews.csv
  Size: 3.96 MB
✓ final_cleaned_movie_reviews.csv
  Size: 111.92 MB
✓ processed_movie_reviews.csv
  Size: 3.96 MB

✓ FINAL CLEANED DATASET EXISTS

STEP 65 COMPLETED


In [111]:
# STEP 66 - VERIFY CLEANED DATASET CONTENT

import pandas as pd

print("=" * 65)
print("STEP 66 - CLEANED DATASET CONTENT VERIFICATION")
print("=" * 65)

# Find the cleaned CSV file
csv_files = list(cleaned_folder.glob("*.csv"))

if len(csv_files) == 0:
    print("❌ No cleaned CSV file found.")

else:
    cleaned_file = csv_files[0]

    # Load cleaned dataset
    final_df = pd.read_csv(cleaned_file)

    print("\nFile:", cleaned_file.name)
    print("Records:", len(final_df))
    print("Columns:", len(final_df.columns))

    print("\nColumn Names:")
    for column in final_df.columns:
        print(" -", column)

    print("\nFirst 5 Records:")
    display(final_df.head())

    print("\nMissing Values:")
    print(final_df.isnull().sum())

    print("\nDuplicate Records:", final_df.duplicated().sum())

    print("\nDataset Information:")
    print(final_df.info())

    print("\n" + "=" * 65)
    print("STEP 66 COMPLETED")
    print("=" * 65)

STEP 66 - CLEANED DATASET CONTENT VERIFICATION

File: cleaned_movie_reviews.csv
Records: 1000
Columns: 3

Column Names:
 - Movie_Name
 - Reviewer
 - Review_Text

First 5 Records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...



Missing Values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate Records: 0

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Movie_Name   1000 non-null   str  
 1   Reviewer     1000 non-null   str  
 2   Review_Text  1000 non-null   str  
dtypes: str(3)
memory usage: 23.6 KB
None

STEP 66 COMPLETED


In [112]:
# STEP 67 - FINAL CLEANED DATASET PREVIEW

import pandas as pd

print("=" * 65)
print("STEP 67 - FINAL CLEANED DATASET PREVIEW")
print("=" * 65)

# Find cleaned CSV
csv_files = list(cleaned_folder.glob("*.csv"))

if len(csv_files) == 0:
    print("❌ No cleaned CSV file found.")
else:
    cleaned_file = csv_files[0]
    final_df = pd.read_csv(cleaned_file)

    print("\nFile:", cleaned_file.name)
    print("Total Records:", len(final_df))
    print("Total Columns:", len(final_df.columns))

    print("\nColumn Names:")
    for column in final_df.columns:
        print("-", column)

    print("\nFirst 5 Records:")
    display(final_df.head())

    print("\nLast 5 Records:")
    display(final_df.tail())

print("\n" + "=" * 65)
print("STEP 67 COMPLETED")
print("=" * 65)

STEP 67 - FINAL CLEANED DATASET PREVIEW

File: cleaned_movie_reviews.csv
Total Records: 1000
Total Columns: 3

Column Names:
- Movie_Name
- Reviewer
- Review_Text

First 5 Records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...



Last 5 Records:


,Movie_Name,Reviewer,Review_Text
995,"Wings of the Dove, The (1997)",reviewed by Scott Renshaw,THE WINGS OF THE DOVE (Miramax) Starring: Hele...
996,Mother (1996/II),reviewed by Brian Takeshita,"I like movies with Albert Brooks, and I really..."
997,"Turning, The (1992)",reviewed by Scott Hamilton & Chris Holland,There's only one conceivable reason for watchi...
998,Halloween 4: The Return of Michael Myers (1988),reviewed by Scott Hamilton & Chris Holland,Halloween 4: The Return of Michael Myers Revie...
999,I Know What You Did Last Summer (1997),reviewed by Tim Voon,"Cast: Jennifer Love Hewitt, Sarah Michelle Gel..."



STEP 67 COMPLETED


In [113]:
# STEP 68 - FINAL DATASET STATISTICS

print("=" * 65)
print("STEP 68 - FINAL DATASET STATISTICS")
print("=" * 65)

print("\nDataset Shape:")
print("Rows:", final_df.shape[0])
print("Columns:", final_df.shape[1])

print("\nColumn Data Types:")
print(final_df.dtypes)

print("\nUnique Values in Each Column:")
for column in final_df.columns:
    print(column, ":", final_df[column].nunique())

print("\nMissing Values:")
print(final_df.isnull().sum())

print("\nDuplicate Records:")
print(final_df.duplicated().sum())

print("\n" + "=" * 65)
print("STEP 68 COMPLETED")
print("=" * 65)

STEP 68 - FINAL DATASET STATISTICS

Dataset Shape:
Rows: 1000
Columns: 3

Column Data Types:
Movie_Name     str
Reviewer       str
Review_Text    str
dtype: object

Unique Values in Each Column:
Movie_Name : 675
Reviewer : 215
Review_Text : 1000

Missing Values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate Records:
0

STEP 68 COMPLETED


In [114]:
# STEP 68 - FINAL DATASET STATISTICS

print("=" * 65)
print("STEP 68 - FINAL DATASET STATISTICS")
print("=" * 65)

print("\nDataset Shape:")
print("Rows:", final_df.shape[0])
print("Columns:", final_df.shape[1])

print("\nColumn Data Types:")
print(final_df.dtypes)

print("\nUnique Values in Each Column:")
for column in final_df.columns:
    print(column, ":", final_df[column].nunique())

print("\nMissing Values:")
print(final_df.isnull().sum())

print("\nDuplicate Records:")
print(final_df.duplicated().sum())

print("\n" + "=" * 65)
print("STEP 68 COMPLETED")
print("=" * 65)

STEP 68 - FINAL DATASET STATISTICS

Dataset Shape:
Rows: 1000
Columns: 3

Column Data Types:
Movie_Name     str
Reviewer       str
Review_Text    str
dtype: object

Unique Values in Each Column:
Movie_Name : 675
Reviewer : 215
Review_Text : 1000

Missing Values:
Movie_Name     0
Reviewer       0
Review_Text    0
dtype: int64

Duplicate Records:
0

STEP 68 COMPLETED


In [115]:
# STEP 69 - FINAL DATA QUALITY VERIFICATION

print("=" * 65)
print("STEP 69 - FINAL DATA QUALITY VERIFICATION")
print("=" * 65)

# 1. Missing values
missing_values = final_df.isnull().sum().sum()

# 2. Duplicate records
duplicate_records = final_df.duplicated().sum()

# 3. Empty text values
empty_text_values = 0

for column in final_df.columns:
    if final_df[column].dtype == "object":
        empty_text_values += (final_df[column].astype(str).str.strip() == "").sum()

# 4. Dataset size
total_rows = len(final_df)
total_columns = len(final_df.columns)

print("\nDataset Quality Results")
print("-" * 65)

print("Total Records       :", total_rows)
print("Total Columns       :", total_columns)
print("Missing Values      :", missing_values)
print("Duplicate Records   :", duplicate_records)
print("Empty Text Values   :", empty_text_values)

print("\nQuality Status")
print("-" * 65)

if missing_values == 0 and duplicate_records == 0 and empty_text_values == 0:
    print("✓ DATASET IS CLEAN")
    print("✓ NO MISSING VALUES")
    print("✓ NO DUPLICATE RECORDS")
    print("✓ NO EMPTY TEXT VALUES")
else:
    print("⚠ DATASET NEEDS FURTHER CLEANING")

print("\n" + "=" * 65)
print("STEP 69 COMPLETED")
print("=" * 65)

STEP 69 - FINAL DATA QUALITY VERIFICATION

Dataset Quality Results
-----------------------------------------------------------------
Total Records       : 1000
Total Columns       : 3
Missing Values      : 0
Duplicate Records   : 0
Empty Text Values   : 0

Quality Status
-----------------------------------------------------------------
✓ DATASET IS CLEAN
✓ NO MISSING VALUES
✓ NO DUPLICATE RECORDS
✓ NO EMPTY TEXT VALUES

STEP 69 COMPLETED


In [116]:
# STEP 70 - SAVE FINAL CLEANED DATASET

final_output_file = cleaned_folder / "PPS1_Final_Cleaned_Movie_Reviews.csv"

final_df.to_csv(final_output_file, index=False, encoding="utf-8-sig")

print("=" * 65)
print("STEP 70 - FINAL CLEANED DATASET SAVED")
print("=" * 65)

print("\nFile Name:", final_output_file.name)
print("Location:", final_output_file)
print("Records:", len(final_df))
print("Columns:", len(final_df.columns))

if final_output_file.exists():
    print("\n✓ FINAL CLEANED DATASET SAVED SUCCESSFULLY")
else:
    print("\n✗ FILE WAS NOT SAVED")

print("\n" + "=" * 65)
print("STEP 70 COMPLETED")
print("=" * 65)

STEP 70 - FINAL CLEANED DATASET SAVED

File Name: PPS1_Final_Cleaned_Movie_Reviews.csv
Location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\PPS1_Final_Cleaned_Movie_Reviews.csv
Records: 1000
Columns: 3

✓ FINAL CLEANED DATASET SAVED SUCCESSFULLY

STEP 70 COMPLETED


In [118]:
# STEP 71 - UPDATE SOURCE URL INFORMATION

from pathlib import Path

source_folder = project_folder / "07_Source_URLs"
source_folder.mkdir(parents=True, exist_ok=True)

source_file = source_folder / "source_urls.txt"

source_content = "\n".join([
    "PPS 1 - MOVIE REVIEW ANALYSIS",
    "DATASET SOURCE INFORMATION",
    "==============================================",
    "",
    "Project:",
    "Dataset Collection, Cleaning and Processing",
    "",
    "Dataset:",
    "Movie User Reviews",
    "",
    "Primary Public Dataset Source:",
    "https://github.com/SimpliSolve/RecSysData",
    "",
    "Repository:",
    "SimpliSolve - RecSysData",
    "",
    "Review Data Folder:",
    "https://github.com/SimpliSolve/RecSysData/tree/master/user_review_data_daily_search",
    "",
    "IMDb Top Movie Information:",
    "https://github.com/SimpliSolve/RecSysData/tree/master/top250",
    "",
    "IMDb Official Website:",
    "https://www.imdb.com/",
    "",
    "Data Origin:",
    "The review records in the public RecSysData repository",
    "originate from IMDb user review pages and were previously",
    "collected by the repository authors.",
    "",
    "Important Note:",
    "This project uses the already-available public dataset",
    "from the repository. It does not perform live scraping",
    "of IMDb during this project.",
    "",
    "Tools Used:",
    "- Python",
    "- Requests",
    "- BeautifulSoup",
    "- Pandas",
    "- Jupyter Notebook",
    "",
    "Processing Performed:",
    "1. Dataset collection",
    "2. Data inspection",
    "3. Missing-value checking",
    "4. Duplicate removal",
    "5. Text cleaning",
    "6. Data validation",
    "7. Final cleaned dataset creation",
    "",
    "Final Output:",
    "PPS1_Final_Cleaned_Movie_Reviews.csv"
])

with open(source_file, "w", encoding="utf-8") as file:
    file.write(source_content)

print("=" * 65)
print("STEP 71 - SOURCE INFORMATION UPDATED")
print("=" * 65)

print("\nFile:", source_file.name)
print("Location:", source_file)

if source_file.exists():
    print("\n✓ SOURCE URL FILE CREATED SUCCESSFULLY")
else:
    print("\n✗ SOURCE URL FILE WAS NOT CREATED")

print("\n" + "=" * 65)
print("STEP 71 COMPLETED")
print("=" * 65)

STEP 71 - SOURCE INFORMATION UPDATED

File: source_urls.txt
Location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\07_Source_URLs\source_urls.txt

✓ SOURCE URL FILE CREATED SUCCESSFULLY

STEP 71 COMPLETED


In [119]:
# STEP 72 - NOTEBOOK FINAL CLEANUP

print("=" * 65)
print("STEP 72 - NOTEBOOK FINAL CLEANUP")
print("=" * 65)

print("\nChecking important project variables...")

checks = {
    "Project Folder": project_folder,
    "Cleaned Folder": cleaned_folder,
    "Final Dataset": final_df,
    "Final Cleaned File": final_output_file,
    "Source URL File": source_file
}

for name, value in checks.items():
    if value is not None:
        print("✓", name)
    else:
        print("✗", name)

print("\nFinal Dataset Information")
print("-" * 65)
print("Records :", len(final_df))
print("Columns :", len(final_df.columns))
print("Missing :", final_df.isnull().sum().sum())
print("Duplicates :", final_df.duplicated().sum())

print("\nNotebook cleanup check completed.")
print("Please save the notebook using Ctrl + S.")

print("\n" + "=" * 65)
print("STEP 72 COMPLETED")
print("=" * 65)

STEP 72 - NOTEBOOK FINAL CLEANUP

Checking important project variables...
✓ Project Folder
✓ Cleaned Folder
✓ Final Dataset
✓ Final Cleaned File
✓ Source URL File

Final Dataset Information
-----------------------------------------------------------------
Records : 1000
Columns : 3
Missing : 0
Duplicates : 0

Notebook cleanup check completed.
Please save the notebook using Ctrl + S.

STEP 72 COMPLETED


In [122]:
# STEP 73 - FINAL PROJECT STATUS CHECK

print("=" * 70)
print("FINAL PROJECT STATUS CHECK")
print("=" * 70)

# Check project folders
required_folders = [
    "01_Raw_Data",
    "02_Cleaned_Data",
    "03_Code",
    "Note_book",
    "05_Documentation",
    "06_Output",
    "07_Source_URLs"
]

all_folders_exist = True

print("\nPROJECT FOLDER CHECK")

for folder_name in required_folders:
    folder_path = project_folder / folder_name

    if folder_path.exists():
        print("✓", folder_name, "- exists")
    else:
        print("✗", folder_name, "- missing")
        all_folders_exist = False


# Dataset quality
print("\n" + "=" * 70)
print("DATASET QUALITY CHECK")
print("=" * 70)

print("Total Records     :", len(final_df))
print("Total Columns     :", len(final_df.columns))
print("Missing Values    :", final_df.isnull().sum().sum())
print("Duplicate Records :", final_df.duplicated().sum())


# Final status
print("\n" + "=" * 70)
print("FINAL PROJECT STATUS")
print("=" * 70)

if (
    all_folders_exist
    and final_df.isnull().sum().sum() == 0
    and final_df.duplicated().sum() == 0
):
    print("✓ ALL PROJECT CHECKS PASSED")
else:
    print("✗ SOME PROJECT CHECKS FAILED")

print("=" * 70)

FINAL PROJECT STATUS CHECK

PROJECT FOLDER CHECK
✓ 01_Raw_Data - exists
✓ 02_Cleaned_Data - exists
✓ 03_Code - exists
✓ Note_book - exists
✓ 05_Documentation - exists
✓ 06_Output - exists
✓ 07_Source_URLs - exists

DATASET QUALITY CHECK
Total Records     : 1000
Total Columns     : 3
Missing Values    : 0
Duplicate Records : 0

FINAL PROJECT STATUS
✓ ALL PROJECT CHECKS PASSED


In [123]:
# STEP 74 - SAVE FINAL CLEANED DATASET

print("=" * 70)
print("STEP 74 - SAVING FINAL CLEANED DATASET")
print("=" * 70)

final_output_file = cleaned_folder / "PPS1_Final_Cleaned_Movie_Reviews.csv"

final_df.to_csv(final_output_file, index=False, encoding="utf-8-sig")

print("\n✓ Final cleaned dataset saved successfully")
print("File:", final_output_file.name)
print("Location:", final_output_file)
print("Records:", len(final_df))
print("Columns:", len(final_df.columns))

print("=" * 70)

STEP 74 - SAVING FINAL CLEANED DATASET

✓ Final cleaned dataset saved successfully
File: PPS1_Final_Cleaned_Movie_Reviews.csv
Location: C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\PPS1_Final_Cleaned_Movie_Reviews.csv
Records: 1000
Columns: 3


In [124]:
# STEP 75 - VERIFY SAVED FINAL DATASET

import pandas as pd

print("=" * 70)
print("STEP 75 - FINAL DATASET VERIFICATION")
print("=" * 70)

# Load the file saved in Step 74
check_df = pd.read_csv(final_output_file)

print("\nFile name:", final_output_file.name)
print("Records:", len(check_df))
print("Columns:", len(check_df.columns))

print("\nColumn Names:")
for column in check_df.columns:
    print("-", column)

print("\nMissing Values:", check_df.isnull().sum().sum())
print("Duplicate Records:", check_df.duplicated().sum())

print("\nFirst 5 Records:")
display(check_df.head())

print("=" * 70)
print("STEP 75 COMPLETED")
print("=" * 70)

STEP 75 - FINAL DATASET VERIFICATION

File name: PPS1_Final_Cleaned_Movie_Reviews.csv
Records: 1000
Columns: 3

Column Names:
- Movie_Name
- Reviewer
- Review_Text

Missing Values: 0
Duplicate Records: 0

First 5 Records:


,Movie_Name,Reviewer,Review_Text
0,"Hitcher, The (1986)",reviewed by Mark R. Leeper,[Editor's note: Sites running 2.10 netnews wil...
1,Beyond Therapy (1987),reviewed by Steve Fritzinger,"Set in the offices of two psychoanalysts, and ..."
2,Angel Heart (1987),reviewed by Clark Quinn,"Starring: Mickey Rourke, Robert DeNiro, Lisa B..."
3,Angel Heart (1987),reviewed by Mark R. Leeper,Harry Angel (played by Mickey Rourke) is a har...
4,Radio Days (1987),reviewed by Jeff Meyer,I've been a little lax in getting out movie re...


STEP 75 COMPLETED


In [126]:
# STEP 76 - FIND CORRECT DATASET

import pandas as pd

print("=" * 70)
print("STEP 76 - SEARCHING FOR DATASET WITH 95,760 RECORDS")
print("=" * 70)

csv_files = list(project_folder.rglob("*.csv"))

print("\nCSV files found:", len(csv_files))

found_dataset = False

for file_path in csv_files:

    try:
        df_check = pd.read_csv(file_path, nrows=5)

        with open(file_path, "r", encoding="utf-8-sig", errors="ignore") as f:
            total_rows = sum(1 for _ in f) - 1

        if total_rows == 95760:

            print("\n✓ MATCH FOUND")
            print("File name :", file_path.name)
            print("Location  :", file_path)
            print("Records   :", total_rows)
            print("Columns   :", len(df_check.columns))
            print("Columns names:")
            print(list(df_check.columns))

            found_dataset = True

    except Exception:
        pass

print("\n" + "=" * 70)

if found_dataset:
    print("✓ DATASET WITH 95,760 RECORDS FOUND")
else:
    print("⚠ NO DATASET WITH EXACTLY 95,760 RECORDS FOUND")

print("=" * 70)

print("\nSTEP 76 COMPLETED")

STEP 76 - SEARCHING FOR DATASET WITH 95,760 RECORDS

CSV files found: 67

✓ MATCH FOUND
File name : final_cleaned_movie_reviews.csv
Location  : C:\Users\vyshn\OneDrive\Desktop\DATA-SCIENCE\PROJECTS\PPS1- Movie review Analysis\02_Cleaned_Data\final_cleaned_movie_reviews.csv
Records   : 95760
Columns   : 8
Columns names:
['username', 'rating', 'helpful', 'total', 'date', 'title', 'review', 'movie_name']

✓ DATASET WITH 95,760 RECORDS FOUND

STEP 76 COMPLETED


In [127]:
# STEP 77 - FINAL PROJECT CHECK

print("=" * 70)
print("PPS 1 - MOVIE REVIEW ANALYSIS")
print("FINAL PROJECT CHECK")
print("=" * 70)

print("\nDATASET INFORMATION")
print("-" * 70)
print("Records :", len(final_df))
print("Columns :", len(final_df.columns))
print("Missing values :", final_df.isnull().sum().sum())
print("Duplicate records :", final_df.duplicated().sum())

print("\nCOLUMN NAMES")
print("-" * 70)

for column in final_df.columns:
    print("-", column)

print("\nFINAL STATUS")
print("-" * 70)

if final_df.isnull().sum().sum() == 0 and final_df.duplicated().sum() == 0:
    print("✓ DATASET IS CLEAN")
    print("✓ NO MISSING VALUES")
    print("✓ NO DUPLICATE RECORDS")
    print("✓ PROJECT DATA PROCESSING COMPLETED")
else:
    print("⚠ PLEASE CHECK DATASET QUALITY")

print("\n" + "=" * 70)
print("✓ PPS 1 PROJECT COMPLETED")
print("=" * 70)

PPS 1 - MOVIE REVIEW ANALYSIS
FINAL PROJECT CHECK

DATASET INFORMATION
----------------------------------------------------------------------
Records : 1000
Columns : 3
Missing values : 0
Duplicate records : 0

COLUMN NAMES
----------------------------------------------------------------------
- Movie_Name
- Reviewer
- Review_Text

FINAL STATUS
----------------------------------------------------------------------
✓ DATASET IS CLEAN
✓ NO MISSING VALUES
✓ NO DUPLICATE RECORDS
✓ PROJECT DATA PROCESSING COMPLETED

✓ PPS 1 PROJECT COMPLETED


In [128]:
# STEP 76 - CHECK AVAILABLE IMDb REVIEW DATA

import requests

api_url = "https://api.github.com/repos/SimpliSolve/RecSysData/contents/user_review_data_daily_search"

response = requests.get(api_url, timeout=30)

print("=" * 70)
print("CHECKING AVAILABLE IMDb MOVIE REVIEW DATA")
print("=" * 70)

print("Status Code:", response.status_code)

if response.status_code == 200:
    items = response.json()

    folders = [
        item["name"]
        for item in items
        if item["type"] == "dir"
    ]

    print("Movie review folders found:", len(folders))

    print("\nFirst 20 movie folders:")
    for folder in folders[:20]:
        print("-", folder)

else:
    print("Could not access the dataset.")

CHECKING AVAILABLE IMDb MOVIE REVIEW DATA
Status Code: 200
Movie review folders found: 39

First 20 movie folders:
- A Worm in the Heart
- Ali's Comeback
- Back to Chernobyl
- Battle of Little Bighorn
- Better
- Bone Detectives Britain's Buried Secrets
- Cannabis vs. Cancer
- Charlie Chaplin, le génie de la liberté
- Cheer
- Couch Connections
- Crip Camp
- Cryptopia Bitcoin, Blockchains and the Future of the Internet
- Days of Rage the Rolling Stones' Road to Altamont
- Fred Jüssi. The Beauty of Being
- Ghost Adventures Screaming Room
- Heroes
- Holy Silence
- In Search of Dracula with Mark Gatiss
- James May Our Man in
- Louis Theroux Selling Sex


In [130]:
# STEP 77 - MATCH IMDb REVIEW MOVIES WITH TOP 100

import re
import pandas as pd

def normalize_title(title):
    title = str(title).lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return " ".join(title.split())

# Normalize Top 100 titles
top100_normalized = {
    normalize_title(title): title
    for title in imdb_top_100
}

# Find matches
matched_movies = []

for folder in folders:
    normalized_folder = normalize_title(folder)

    if normalized_folder in top100_normalized:
        matched_movies.append({
            "Top_100_Title": top100_normalized[normalized_folder],
            "Review_Folder": folder
        })

matched_df = pd.DataFrame(matched_movies)

print("=" * 70)
print("STEP 77 - TOP 100 MATCHING")
print("=" * 70)

print("Total review folders:", len(folders))
print("Top 100 movies matched:", len(matched_df))

print("\nMatched movies:")

if len(matched_df) > 0:
    print(matched_df.to_string(index=False))
else:
    print("No Top 100 movies matched.")

STEP 77 - TOP 100 MATCHING
Total review folders: 39
Top 100 movies matched: 0

Matched movies:
No Top 100 movies matched.


In [131]:
# STEP 78 - CHECK MOVIE COVERAGE IN IMDb REVIEW DATA

import pandas as pd

print("=" * 70)
print("STEP 78 - IMDb REVIEW DATA MOVIE COVERAGE")
print("=" * 70)

print("Total review records:", len(imdb_reviews_df))
print("Unique IMDb movie IDs:", imdb_reviews_df["movie_id"].nunique())

print("\nReviews per movie:")
movie_counts = (
    imdb_reviews_df
    .groupby("movie_id")
    .size()
    .sort_values(ascending=False)
)

print(movie_counts.head(20))

print("\n" + "=" * 70)
print("MOVIE COVERAGE CHECK COMPLETED")
print("=" * 70)

STEP 78 - IMDb REVIEW DATA MOVIE COVERAGE
Total review records: 113538
Unique IMDb movie IDs: 146

Reviews per movie:
movie_id
tt0111161     9184
tt0816692     6787
tt0944947     6015
tt5180504     5783
tt7631058     5608
tt0903747     5342
tt7462410     4201
tt10919420    3248
tt10888594    2575
tt11198330    2372
tt0241527     2074
tt6277462     1914
tt7556122     1814
tt1751634     1612
tt7608028     1525
tt2306299     1465
tt0185183     1445
tt3566834     1437
tt4123432     1427
tt10886166    1422
dtype: int64

MOVIE COVERAGE CHECK COMPLETED


In [132]:
# STEP 79 - CHECK AVAILABLE MOVIE ID MAPPING

print("=" * 70)
print("STEP 79 - MOVIE ID MAPPING CHECK")
print("=" * 70)

print("Review dataset movie IDs:", imdb_reviews_df["movie_id"].nunique())

print("\nMovie IDs already available in our Top 250 source:",
      top250_df["movie_id"].nunique())

# Find overlap
review_ids = set(imdb_reviews_df["movie_id"].dropna())
known_ids = set(top250_df["movie_id"].dropna())

overlap_ids = review_ids.intersection(known_ids)

print("Movie IDs that we can currently identify:", len(overlap_ids))

print("\nSample identified movies:")
print(
    top250_df[
        top250_df["movie_id"].isin(list(overlap_ids)[:10])
    ][["movie_id", "movie_name"]].to_string(index=False)
)

print("\n" + "=" * 70)
print("MAPPING CHECK COMPLETED")
print("=" * 70)

STEP 79 - MOVIE ID MAPPING CHECK
Review dataset movie IDs: 146

Movie IDs already available in our Top 250 source: 50
Movie IDs that we can currently identify: 2

Sample identified movies:
 movie_id               movie_name
tt0111161 The Shawshank Redemption
tt0816692             Interstellar

MAPPING CHECK COMPLETED
